# ML-05 · Образовательная миграция ВО и СПО

**«Демографический советник» · Этап II · `stage2-ml5-v2.0.2`**

Единый самостоятельный Colab-ноутбук. Источники — **только** неизменённый `handoff/07_v2/dataset_index.json` приватного репозитория Этапа I, его штатные исходные таблицы и источниковые манифесты. Другие ML-исследования не читаются, не импортируются и не изменяются.

**Семь проверок:** (1) потоки между регионами; (2) приём в учреждения; (3) образовательный отток; (4) программы; (5) трудоустройство; (6) межрегиональная сеть; (7) муниципальная демография: E0 против E1. При нехватке данных применяется `NOT_FEASIBLE`, не создаются синтетические потоки, география трудоустройства или фиктивные нули.

**Статус выдаваемого файла:** разработан и статически проверен; **не** является исполненным научным исследованием. `SMOKE` проверяет воспроизводимую загрузку/код, `FULL` исполняет научный протокол. Публикация запрещена до `FULL` и успешных проверок всех применимых компонентов.

**Инструкция:** откройте этот файл в Colab, укажите токен с правом чтения приватного репозитория в Colab Secrets как `GITHUB_TOKEN`, режим `MODE="FULL"` уже выбран в конфигурации; запустите **Runtime → Run all**. Google Drive не требуется.

## 1. Паспорт, гипотезы и правило научной приёмки

Образовательная мобильность означает измеренное различие между территорией предыдущего образования и территорией обучения; она **не равна** постоянному миграционному учёту. `origin_residence`, `origin_previous_education`, `destination_study`, `employment_location` и `postgraduation_residence` считаются разными величинами. Данные о трудоустройстве и заработке не доказывают закрепление выпускника. Учреждения и филиалы сохраняют собственные географические ключи; юридический адрес головной организации не приписывается филиалу.

H5.1 — модель улучшает прогноз подтверждённых OD-потоков относительно инерции; H5.2 — характеристики организации помогают прогнозировать приём; H5.3 — образовательный отток неоднороден; H5.4 — программы различаются по географии выбора; H5.5 — характеристики связаны с трудоустройством; H5.6 — регионы образуют устойчивые сетевые центры; H5.7 — образовательная инфраструктура улучшает муниципальный прогноз внеэкономических факторов. **Ни одна гипотеза не считается подтверждённой до независимого теста.**

## 2. Единая конфигурация

Режим `FULL` выбран; первый безопасный запуск можно сделать с `MODE="SMOKE"`. Правьте **только эту ячейку**. `SMOKE` не публикует результаты и не приравнивается к научному прогону; `FULL` не публикует результаты, если не выполнен полный научный QA. Идентификаторы показателей выбираются **только по их точным оригинальным названиям** после просмотра автоматически выгруженного словаря возможных показателей; неоднозначные названия не выбираются догадкой.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass(frozen=True)
class Config:
    MODE: str = "FULL"                        # "SMOKE" или "FULL"
    REPO: str = "AntonDozhdikov/Demograf_sovetnik"
    INPUT_REF: str = "main"
    INDEX: str = "handoff/07_v2/dataset_index.json"
    LOADER_REFERENCE: str = "handoff/07_v2/stage2_loader.py"
    SEGMENT: str = "stage2/ml05_education"
    VERSION: str = "stage2-ml5-v2.0.2"
    TARGET_BRANCH: str = "ml05-education-20261102T160726Z"  # существующая собственная ветка, заменяем только сегмент
    EXPECT_OLD_VERSION: str = "stage2-ml5-v2.0.1"
    OVERWRITE_OWN_SEGMENT: bool = True
    NETWORK_EXCLUDE_DIAGONAL: bool = True
    STRICT_SEMANTIC_GATE: bool = True  # предупреждение о непроверенной семантике E01
    REQUIRE_NONDETERIORATION: bool = False  # отрицательные результаты сохраняются; качество не имитируется
    RUN_EDGE_COVERAGE_AUDIT: bool = True
    ROOT: str = "/content/ml05_education_v202"
    PUBLISH_AFTER_FULL_QA: bool = True  # если FULL, заменяет файлы ТОЛЬКО после всех QA
    HORIZONS: tuple = (1,)
    DEV_SEEDS: tuple = (17, 29, 41, 53, 67, 79, 83, 97, 101, 113)
    FINAL_SEEDS: tuple = tuple(range(1001,1051))
    MAX_FINAL_SEEDS_SMOKE: int = 1
    BOOTSTRAP_REPS: int = 400
    MIN_TRAIN_ROWS: int = 80
    MIN_TEST_ROWS: int = 30
    MIN_YEARS: int = 4
    MIN_MUNI_E1_SUPPORT: int = 60
    PUBLICATION_LAG_YEARS: int = 1
    MAX_ASSET_MB: int = 900
    SMOKE_ASSET_LIMIT: int = 4
    # Заполняются после содержательной проверки справочника indicator_name.
    APPROVED_E02_ADMISSIONS: tuple = ()          # точные названия E02 индикаторов приёма
    APPROVED_SPO_ADMISSIONS: tuple = ()          # точные названия СПО индикаторов приёма
    APPROVED_VPO_ADMISSIONS: tuple = ()          # точные названия ВПО-1 приёма, если обоснованы
    ALLOW_CONDITIONAL_OD_SEMANTICS: bool = False # если не определено основание происхождения, OD только как условная мобильность
    NETWORK_MIN_YEARS: int = 2

CFG = Config()
assert CFG.MODE in {"SMOKE", "FULL"}
OUT = Path(CFG.ROOT)
OUT.mkdir(parents=True,exist_ok=True)
(OUT/"inputs").mkdir(exist_ok=True)
(OUT/"derived").mkdir(exist_ok=True)
(OUT/"figures").mkdir(exist_ok=True)
print(f"ML-05 · режим {CFG.MODE} · версия {CFG.VERSION} · сегмент {CFG.SEGMENT}")

## 3–4. Диагностика Colab и минимальные зависимости

GPU для базовых методов не требуется. Установки выполняются только для отсутствующих пакетов; сообщения pip и предупреждения о версиях не являются доказательством корректности расчётов.

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil, time, hashlib, json, traceback, re
from datetime import datetime, timezone
CORE={"numpy":"numpy", "pandas":"pandas", "pyarrow":"pyarrow", "duckdb":"duckdb", "sklearn":"scikit-learn", "scipy":"scipy", "catboost":"catboost", "networkx":"networkx", "matplotlib":"matplotlib", "xlsxwriter":"XlsxWriter", "nbformat":"nbformat", "requests":"requests", "tqdm":"tqdm"}
missing=[p for m,p in CORE.items() if importlib.util.find_spec(m) is None]
if missing:
    proc=subprocess.run([sys.executable,"-m","pip","install","-q",*missing],capture_output=True,text=True)
    if proc.returncode: raise RuntimeError("Ошибка установки библиотек: "+", ".join(missing))
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, duckdb, requests, nbformat, networkx as nx
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
from tqdm.auto import tqdm
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
RUN_STARTED=time.monotonic()
DISK=shutil.disk_usage(CFG.ROOT)
print({"python":sys.version.split()[0],"cpu":os.cpu_count(),"free_gb":round(DISK.free/1e9,2),"ram_gb":round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1e9,2) if hasattr(os,'sysconf') else None,"gpu_optional":True})

## 5. Защищённое чтение handoff Этапа I

Аутентификация: Colab Secrets (`GITHUB_TOKEN`) или переменная окружения. Токен не записывается в выходные файлы. Для `git`- и `release`-объектов применяется ровно транспорт, заданный их `locator` в индексе; каждый скачанный файл проверяется по SHA-256. До успешного научного прогона записи в GitHub нет.

In [ ]:
import base64, getpass
from urllib.parse import quote

def get_token():
    token=os.environ.get("GITHUB_TOKEN","").strip()
    if not token:
        try:
            from google.colab import userdata
            token=(userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception: pass
    if not token: token=getpass.getpass("GitHub token (ввод скрыт): ").strip()
    if not token: raise RuntimeError("BLOCKED_AUTH: необходим GITHUB_TOKEN для приватного Stage I")
    return token
TOKEN=get_token()
API="https://api.github.com/repos/"+CFG.REPO
SESSION=requests.Session()
SESSION.headers.update({"Authorization":"Bearer "+TOKEN,"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"})

def safe_request(method,url,**kwargs):
    for retry in range(4):
        try:
            r=SESSION.request(method,url,timeout=kwargs.pop("timeout",(30,600)),**kwargs)
            if r.status_code in (429,502,503,504):
                r.close(); time.sleep(2**retry);continue
            if r.status_code in (401,403,404):
                status=r.status_code;r.close()
                raise PermissionError(f"GitHub {status}: не найден файл или недостаточно прав к приватному репозиторию")
            r.raise_for_status()
            return r
        except PermissionError: raise
        except requests.RequestException:
            if retry==3: raise RuntimeError("Сетевая ошибка при получении данных Stage I") from None
            time.sleep(2**retry)
    raise RuntimeError("GitHub: исчерпаны повторы")

def git_text(path,ref=CFG.INPUT_REF):
    u=API+"/contents/"+quote(path,safe="/")
    r=safe_request("GET",u,params={"ref":ref});data=r.json();r.close()
    if not isinstance(data,dict) or not data.get("content"):
        raise RuntimeError("Недоступно содержимое файла из GitHub Contents: "+path)
    return base64.b64decode(data["content"]).decode("utf-8")

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""):h.update(chunk)
    return h.hexdigest()

def download_asset(a):
    loc=a["locator"]; target=OUT/"inputs"/(hashlib.sha256(a["asset_id"].encode()).hexdigest()[:20]+".parquet")
    if target.exists() and sha256(target)==a["sha256"]:return target
    target.unlink(missing_ok=True)
    if loc["type"]=="git":
        u=API+"/contents/"+quote(loc["path"],safe="/")
        r=safe_request("GET",u,params={"ref":CFG.INPUT_REF})
        payload=r.json();r.close()
        if payload.get("content"):
            target.write_bytes(base64.b64decode(payload["content"]))
        else:
            rr=safe_request("GET",u,params={"ref":CFG.INPUT_REF},headers={"Accept":"application/vnd.github.raw+json"},stream=True)
            with open(target,"wb") as f:
                for part in rr.iter_content(2*1024*1024):
                    if part:f.write(part)
            rr.close()
    elif loc["type"]=="release":
        u=API+"/releases/assets/"+str(loc["asset_id"])
        r=safe_request("GET",u,headers={"Accept":"application/octet-stream"},stream=True)
        if "json" in r.headers.get("content-type","").lower():
            detail=r.json();r.close()
            r=safe_request("GET",detail["browser_download_url"],stream=True)
        with open(target,"wb") as f:
            for part in r.iter_content(2*1024*1024):
                if part:f.write(part)
        r.close()
    else: raise RuntimeError("Неизвестный транспорт Stage I: "+str(loc["type"]))
    actual=sha256(target)
    if actual!=a["sha256"]:
        target.unlink(missing_ok=True)
        raise RuntimeError("SHA256 не совпадает: "+a["asset_id"])
    return target

INDEX=json.loads(git_text(CFG.INDEX))
assert INDEX.get("version")=="7.2", "Версия handoff отличается: проверьте схему"
assert INDEX.get("repo")==CFG.REPO
ASSETS=INDEX["assets"]
assert len(ASSETS)>=1 and all(a.get("sha256_verified") for a in ASSETS)
assert all(len(a.get("sha256", ""))==64 for a in ASSETS)
reference=git_text(CFG.LOADER_REFERENCE)
SOURCE_MANIFESTS={segment:json.loads(git_text(INDEX["source_manifests"][segment])) for segment in ["03","04","05"]}
assert "fetch_asset" in reference and "releases/assets" in reference, "Изменился референсный протокол"
print(f"PASS_INPUT_CATALOG: версия {INDEX['version']}; {len(ASSETS)} объектов; схема и способ загрузки подтверждены")

## 6. DATA_GATE: каталог, фактические словари и определения

Сначала сопоставляются реальные поля исходников; индикаторные строки без подтверждённого смысла не преобразуются в зачисленных, образовательные мигранты или закрепившихся. `edu_od_year` по E01 относится к СПО, но сама схема не сообщает, измеряет ли происхождение место предыдущего образования или регистрацию. Поэтому термин остаётся «источник E01», пока не прочитано определение первичного показателя. ВО не получает подставленную матрицу СПО.

In [ ]:
from collections import defaultdict
ASSET_MAP={a["asset_id"]:a for a in ASSETS}

def choose(segment=None,role=None,source=None,prefix=None):
    return [a for a in ASSETS if (segment is None or a["segment"]==segment)
            and (role is None or a.get("role")==role)
            and (source is None or a.get("source_id")==source)
            and (prefix is None or a["asset_id"].startswith(prefix))]

def exactly_one(candidates,label):
    if len(candidates)!=1:raise RuntimeError(f"DATA_GATE: {label}: найдено {len(candidates)} вместо 1")
    return candidates[0]

def asset_df(a,columns=None,where=None,limit=None):
    path=download_asset(a)
    fields=set(a.get("columns_manifest") or [])
    if columns and fields and not set(columns).issubset(fields):
        raise RuntimeError("Неверные поля "+a["asset_id"]+": "+str(set(columns)-fields))
    con=duckdb.connect(database=":memory:")
    try:
        if columns:
            # имена столбцов берутся только из подтверждённого манифеста
            cols=", ".join('"'+c.replace('"','""')+'"' for c in columns)
        else: cols="*"
        sql=f"SELECT {cols} FROM read_parquet(?)"
        if where: sql+=" WHERE "+where  # where передаётся только локальными фиксированными выражениями
        if limit:sql+=" LIMIT "+str(int(limit))
        return con.execute(sql,[str(path)]).df()
    finally:con.close()

candidates={
 "spo_od": choose("03",role="education_od_flows"),
 "spo_registry": choose("03",role="education_organizations"),
 "spo_org": sorted(choose("03",role="organization_indicators"),key=lambda a:a["rows_actual"],reverse=True),
 "spo_program": choose("03",role="regional_spo"),
 "vo_e02": choose("04",source="E02"),
 "vo_vpo_admission":choose("04",source="VPO_2_1_1"),
 "graduates":choose("05",role="organization_total"),
 "mun":choose("01",source="D01"),
 "spatial":choose("06",role="derived"),
}
ASSET_AUDIT=pd.DataFrame([{"asset_id":a["asset_id"],"segment":a["segment"],"role":a.get("role",""),"source":a["source_id"],"rows_catalog":a.get("rows_actual"),"sha256":a["sha256"],"fields":", ".join(a.get("columns_manifest",[])[:30])} for a in ASSETS])
STATUS={k:{"status":"PENDING","reason":"Данные пока не анализировались"} for k in [f"ML-5.{i}" for i in range(1,8)]}
ISSUES=[];PROVENANCE=[];PANELS={};QA=[];PREDICTIONS=[];METRICS=[];SPLITS=[];FIGURES=[];MODELS_META=[];MODEL_FILES={};ACTIVE_MODULE=None

def issue(module,reason,missing=""):
    ISSUES.append({"module":module,"status":"NOT_FEASIBLE","reason":reason,"required_fields":missing})
    STATUS[module]={"status":"NOT_FEASIBLE","reason":reason}

def gate(name,df,keys=None):
    if df is None or df.empty:
        QA.append({"table":name,"status":"EMPTY","rows":0});return False
    bad=0
    if keys:
        if not set(keys).issubset(df.columns):raise AssertionError(f"{name}: нет ключей {set(keys)-set(df.columns)}")
        bad=int(df.duplicated(keys).sum())
    QA.append({"table":name,"status":"PASS" if bad==0 else "DUPLICATE_KEYS","rows":len(df),"duplicates":bad,"years":df["year"].nunique() if "year" in df.columns else None})
    return bad==0

def add_provenance(df,a,panel,semantics):
    if df is not None and not df.empty:
        PROVENANCE.append({"panel":panel,"asset_id":a["asset_id"],"source_id":a["source_id"],"source_sha256":a["sha256"],"rows":len(df),"definition":semantics})

assert len(candidates['spo_od'])==1, 'Нет однозначной таблицы СПО OD'
print("DATA_GATE: источники по ролям:", {k:len(v) for k,v in candidates.items()})

## 7. Шесть нормализованных панелей без неявных JOIN

Таблицы сохраняют собственные ключи. В показателях `year` — год, заявленный источником; год публикации хранится отдельно. Отсутствие записи OD не означает ноль. Экономические и региональные величины не раздаются муниципалитетам через `region` как будто это наблюдения муниципалитета. При несовпадении идентификаторов выполняется карантин, а не нечёткое массовое объединение.

In [ ]:
def sql_asset(a,sql):
    path=download_asset(a)
    con=duckdb.connect(database=":memory:")
    try:return con.execute(sql,[str(path)]).df()
    finally:con.close()

def normyear(x):return pd.to_numeric(x,errors="coerce").astype("Int64")
def normcode(x):return x.astype("string").str.strip().str.replace(r"\.0$","",regex=True)
def numeric(x):return pd.to_numeric(x,errors="coerce")

# 7a. Наблюдаемая OD-сеть СПО — без добавления необнаруженных рёбер.
spo_od_asset=exactly_one(candidates["spo_od"],"SPО OD")
od=asset_df(spo_od_asset,["year","origin_region_code","destination_region_code","origin_region_name","destination_region_name","indicator_value"])
od=od.rename(columns={"indicator_value":"flow_count","origin_region_code":"origin_region","destination_region_code":"destination_region"})
od["year"]=normyear(od["year"])
for c in ("origin_region","destination_region"):od[c]=normcode(od[c])
od["flow_count"]=numeric(od["flow_count"])
od["sector"]="СПО";od["source_definition"]="E01: источник предыдущего образования/проживания уточняется по первоисточнику"
od=od.loc[od.year.notna()&od.origin_region.notna()&od.destination_region.notna()&od.flow_count.notna()].copy()
od=od.loc[(od.flow_count>=0)&np.isclose(od.flow_count,np.round(od.flow_count))].copy()
OD_KEYS=["origin_region","destination_region","year","sector"]
if not gate("edu_od_year",od,OD_KEYS):
    dup=od.loc[od.duplicated(OD_KEYS,keep=False)].copy()
    PANELS["edu_od_ambiguous"]=dup
    od=od.loc[~od.duplicated(OD_KEYS,keep=False)]
    ISSUES.append({"module":"ML-5.1","status":"WARN","reason":"Неоднозначные OD-дубли исключены, не суммированы","required_fields":"доказательство непересечения категорий"})
PANELS["edu_od_year"]=od
add_provenance(od,spo_od_asset,"edu_od_year","наблюдаемые OD-значения СПО; нулевые рёбра не достраиваются")

# 7b. Муниципальная демография D01. ОКТМО — строка.
a_mun=exactly_one(candidates["mun"],"D01 муниципалитеты")
mun=asset_df(a_mun,["oktmo","region","municipality","mun_type","year","population","migration","migration_rate"])
mun["oktmo"]=normcode(mun["oktmo"]);mun["year"]=normyear(mun["year"])
for c in ["population","migration","migration_rate"]:mun[c]=numeric(mun[c])
mun=mun.dropna(subset=["oktmo","year"]).copy()
if not gate("municipal_demography",mun,["oktmo","year"]):
    PANELS["municipal_duplicates"]=mun.loc[mun.duplicated(["oktmo","year"],keep=False)]
    mun=mun.loc[~mun.duplicated(["oktmo","year"],keep=False)]
PANELS["municipal_demography"]=mun
add_provenance(mun,a_mun,"municipal_demography","муниципальная демография D01, не отраслевое OD")

# 7c. Реестр ВО E02 с подтверждаемыми id и parent_id; медленно меняющиеся атрибуты.
if CFG.MODE=="FULL":
    a_e02=exactly_one(candidates["vo_e02"],"E02 ВО")
    q='SELECT DISTINCT "id", "parent_id", "name", "parent_name", "region", "oktmo", "year", "institution" FROM read_parquet(?) WHERE "id" IS NOT NULL AND "year" IS NOT NULL'
    vo_reg=sql_asset(a_e02,q)
    vo_reg["year"]=normyear(vo_reg.year)
    for c in ["id","parent_id","oktmo"]:vo_reg[c]=normcode(vo_reg[c])
    vo_reg["entity_id"]="ВО:"+vo_reg["id"].astype(str)
    vo_reg["group_id"]=np.where(vo_reg.parent_id.notna()&(vo_reg.parent_id!=""), "ВО:"+vo_reg.parent_id.astype(str),vo_reg.entity_id)
    vo_reg["sector"]="ВО"
    inconsistent=vo_reg.duplicated(["entity_id","year"],keep=False)
    PANELS["vo_registry_quarantine"]=vo_reg.loc[inconsistent].copy()
    vo_reg=vo_reg.loc[~inconsistent].copy()
    gate("vo_registry",vo_reg,["entity_id","year"])
    add_provenance(vo_reg,a_e02,"edu_org_registry_vo","реестр E02; точка расположения подтверждается отдельно")
else:vo_reg=pd.DataFrame()

# 7d. Реестр СПО: id и head сохраняются как исходные строки, тип связи не угадывается.
if CFG.MODE=="FULL":
    a_sporeg=exactly_one(candidates["spo_registry"],"реестр СПО")
    cols=["year","region_name","region_oktmo","name","city","lat","lon","type","id","address","head","branches"]
    spo_reg=asset_df(a_sporeg,cols).drop_duplicates()
    spo_reg["year"]=normyear(spo_reg.year)
    spo_reg["id"]=normcode(spo_reg["id"])
    spo_reg["entity_id"]="СПО:"+spo_reg["id"].fillna("НЕИЗВЕСТНО")
    spo_reg["group_id"]=spo_reg["entity_id"]
    spo_reg["sector"]="СПО";spo_reg["match_confidence"]="id_from_source_not_cross_registry"
    bad=spo_reg.duplicated(["entity_id","year"],keep=False)
    PANELS["spo_registry_quarantine"]=spo_reg.loc[bad].copy()
    spo_reg=spo_reg.loc[~bad].copy()
    gate("spo_registry",spo_reg,["entity_id","year"])
    add_provenance(spo_reg,a_sporeg,"edu_org_registry_spo","реестр организаций E01; адрес площадки не доказан для всех филиалов")
else:spo_reg=pd.DataFrame()
PANELS["edu_org_registry"]=pd.concat([vo_reg,spo_reg],ignore_index=True) if CFG.MODE=="FULL" else pd.DataFrame()

# 7e. Возможные индикаторы зачисления: словарь исходных формулировок, без выбора по regex.
if CFG.MODE=="FULL":
    a_spo_ind=candidates["spo_org"][0]
    spo_indicator_dictionary=sql_asset(a_spo_ind,'SELECT DISTINCT "indicator_name", "indicator_section", "indicator_unit" FROM read_parquet(?) WHERE "indicator_name" IS NOT NULL')
    e02_indicator_dictionary=sql_asset(a_e02,'SELECT DISTINCT "indicator_name", "indicator_unit" FROM read_parquet(?) WHERE "indicator_name" IS NOT NULL')
else:
    spo_indicator_dictionary=pd.DataFrame();e02_indicator_dictionary=pd.DataFrame()
PANELS["spo_indicator_dictionary"]=spo_indicator_dictionary
PANELS["e02_indicator_dictionary"]=e02_indicator_dictionary

def exact_indicators(actual,approved,label):
    approved=set(approved)
    if not approved:return False
    unknown=approved-set(actual.astype(str).tolist())
    if unknown:raise ValueError(f"{label}: нет точных индикаторов в первоисточнике: {sorted(unknown)}")
    return True

# 7f. Реальная панель учреждений×год только для однозначно выбранного показателя.
org_parts=[]
if CFG.MODE=="FULL" and exact_indicators(e02_indicator_dictionary.indicator_name,CFG.APPROVED_E02_ADMISSIONS,"E02"):
    chosen=CFG.APPROVED_E02_ADMISSIONS
    # Несколько строк одного индикатора (например, по специальностям) НЕ суммируем без доказательства разбиения.
    raw=asset_df(a_e02,["id","year","indicator_name","area","specialty_code","indicator_value"])
    raw=raw.loc[raw.indicator_name.isin(chosen)].copy()
    raw=raw.loc[raw.area.isna()&raw.specialty_code.isna()]
    raw["year"]=normyear(raw.year);raw["entity_id"]="ВО:"+normcode(raw.id).astype(str)
    raw["admissions"]=numeric(raw.indicator_value)
    raw=raw.dropna(subset=["admissions","year"])
    dup=raw.duplicated(["entity_id","year"],keep=False)
    PANELS["vo_admission_quarantine"]=raw.loc[dup]
    raw=raw.loc[~dup,["entity_id","year","admissions"]].copy()
    raw=raw.merge(vo_reg[["entity_id","year","group_id","region"]],on=["entity_id","year"],validate="one_to_one")
    raw["sector"]="ВО";org_parts.append(raw)
if CFG.MODE=="FULL" and exact_indicators(spo_indicator_dictionary.indicator_name,CFG.APPROVED_SPO_ADMISSIONS,"E01"):
    a=candidates["spo_org"][0]
    raw=asset_df(a,["name","region_name","year","indicator_name","org_value"])
    raw=raw.loc[raw.indicator_name.isin(CFG.APPROVED_SPO_ADMISSIONS)].copy()
    raw["year"]=normyear(raw.year);raw["admissions"]=numeric(raw.org_value)
    raw=raw.dropna(subset=["admissions","year"])
    # Сопоставление name×region×year только если 1:1 с устойчивым реестром.
    keys=["name","region_name","year"]
    lookup=spo_reg[keys+["entity_id","group_id"]].drop_duplicates()
    amb=lookup.duplicated(keys,keep=False)
    PANELS["spo_match_quarantine"]=lookup.loc[amb]
    lookup=lookup.loc[~amb]
    raw=raw.merge(lookup,on=keys,how="left",validate="many_to_one",indicator="_match")
    PANELS["spo_unmatched_indicators"]=raw.loc[raw._match!="both"].copy()
    raw=raw.loc[raw._match=="both"]
    dup=raw.duplicated(["entity_id","year"],keep=False)
    PANELS["spo_admission_quarantine"]=raw.loc[dup]
    raw=raw.loc[~dup,["entity_id","year","admissions","group_id","region_name"]].rename(columns={"region_name":"region"})
    raw["sector"]="СПО";org_parts.append(raw)
org_year=pd.concat(org_parts,ignore_index=True) if org_parts else pd.DataFrame()
PANELS["edu_org_year"]=org_year
gate("edu_org_year",org_year,["entity_id","year","sector"] if not org_year.empty else None)

# 7g. Программы: исходные коды без переприсваивания неизвестного crosswalk.
if CFG.MODE=="FULL":
    a_prog=exactly_one(candidates["spo_program"],"региональные программы СПО")
    program=asset_df(a_prog,["year","region_name","specialty_code","specialty_name","indicator_name","indicator_unit","region_value"])
    program["year"]=normyear(program.year);program["specialty_code"]=normcode(program.specialty_code)
    program["value"]=numeric(program.region_value)
    program=program.dropna(subset=["year","specialty_code","value"])
    program["sector"]="СПО";program["programme_family_unverified"]=program.specialty_code.str.slice(0,2)
    add_provenance(program,a_prog,"edu_program_year","названия программ и коды СПО в первичном классификаторе; crosswalk не утверждён")
else:program=pd.DataFrame()
PANELS["edu_program_year"]=program

# 7h. Выпускники: год в E04 не равен году наблюдения занятости.
if CFG.MODE=="FULL":
    a_grad=exactly_one(candidates["graduates"],"результаты выпускников")
    grad=asset_df(a_grad,["object_level","object_name","oktmo","gender","education_level","year","university","count_graduate","percent_employed","average_salary_fact_med"])
    grad["year"]=normyear(grad.year);grad["oktmo"]=normcode(grad.oktmo)
    for col in ["count_graduate","percent_employed","average_salary_fact_med"]:grad[col]=numeric(grad[col])
    grad["employment_reference_year"]=int(SOURCE_MANIFESTS["05"]["files"]["E04::organization_total"]["employment_reference"][:4])
    grad["measurement_definition"]="E04: employment_reference=2024-12-31; год исходной колонки year НЕ интерпретирован автоматически как год измерения"
    add_provenance(grad,a_grad,"graduate_outcomes","занятость и зарплата на дату референса манифеста 2024-12-31; без места работы")
else:grad=pd.DataFrame()
PANELS["graduate_outcomes"]=grad

print("Шесть логических панелей созданы/проверены:",{k:len(PANELS.get(k,[])) for k in ["edu_org_registry","edu_org_year","edu_od_year","edu_program_year","graduate_outcomes","municipal_demography"]})

## 7.1. Межрегиональные и внутрирегиональные значения OD

Диагональ исключается из прогнозов межрегионального оттока и пространственных сетей. Неизвестные коды сохраняются отдельной таблицей; отсутствие ребра **не** приравнивается к нулю.

In [ ]:
# Исправление v2.0.1: ни один регион внутри себя не является межрегиональным OD.
# Коды неизвестных территорий не превращаются в настоящие регионы и не попадают в отток.
def unknown_region(series):
    z=series.astype('string').str.strip()
    return z.isna()|z.eq('')|z.str.fullmatch(r'0+').fillna(False)|z.str.lower().isin(['nan','none','null','неизвестно'])

def split_od(raw):
    all_od=raw.copy()
    ko=~unknown_region(all_od.origin_region)
    kd=~unknown_region(all_od.destination_region)
    known=all_od.loc[ko&kd].copy()
    unknown=all_od.loc[~(ko&kd)].copy()
    intra=known.loc[known.origin_region==known.destination_region].copy()
    inter=known.loc[known.origin_region!=known.destination_region].copy()
    assert len(all_od)==len(unknown)+len(intra)+len(inter)
    assert (inter.origin_region!=inter.destination_region).all()
    assert not unknown_region(inter.origin_region).any() and not unknown_region(inter.destination_region).any()
    return inter,intra,unknown

od_inter,od_intra,od_unknown=split_od(od)
PANELS['edu_od_interregional']=od_inter
PANELS['edu_od_intraregional']=od_intra
PANELS['edu_od_unknown_region']=od_unknown
assert gate('edu_od_interregional',od_inter,OD_KEYS), 'Дубли межрегиональных OD'
comp=[]
for y in sorted(od.year.dropna().unique()):
    parts=[('ВСЕ_ЗАРЕГИСТРИРОВАННЫЕ',od),('МЕЖРЕГИОНАЛЬНЫЕ',od_inter),('ВНУТРИРЕГИОНАЛЬНЫЕ',od_intra),('НЕОПРЕДЕЛЁННЫЕ_КОДЫ',od_unknown)]
    for kind,d in parts:
        q=d.loc[d.year==y]
        comp.append({'year':int(y),'category':kind,'rows':len(q),'observed_count':float(q.flow_count.sum())})
PANELS['od_composition']=pd.DataFrame(comp)
OD_INTER_PRE_FLIGHT={
    'all_rows':len(od),'inter_rows':len(od_inter),'intra_rows':len(od_intra),
    'unknown_rows':len(od_unknown),'inter_value':float(od_inter.flow_count.sum()),
    'intra_value':float(od_intra.flow_count.sum()),'unknown_value':float(od_unknown.flow_count.sum())}
assert len(od_inter)>0, 'Нет наблюдаемых реальных межрегиональных рёбер'
ISSUES.append({'module':'ML-5.1/3/6','status':'SCIENTIFIC_LIMITATION',
               'reason':'E01: вид предыдущего региона требует сверки первоисточника; межрегиональный переход не означает постоянную смену жительства',
               'required_fields':'source definition origin_previous_education/origin_residence'})
print('OD · зарегистрировано / межрегиональные / диагональ / неизвестные:',
      len(od),len(od_inter),len(od_intra),len(od_unknown))


# Динамика реально зарегистрированных рёбер: не считать пропуск в выгрузке нулём.
def edge_coverage_audit(data):
    k=['origin_region','destination_region','sector']
    t=data[k+['year','flow_count']].copy(); t['year']=t.year.astype(int)
    yrs=sorted(t.year.unique()); details=[]; annual=[]
    for y in yrs:
        now=t[t.year==y].copy(); nowkey=set(map(tuple,now[k].astype(str).to_numpy()))
        prev=t[t.year==y-1].copy(); prevkey=set(map(tuple,prev[k].astype(str).to_numpy()))
        common=nowkey & prevkey; new=nowkey-prevkey; absent=prevkey-nowkey
        annual.append({'year':y,'observed_edges':len(nowkey),'observed_people':float(now.flow_count.sum()),
                       'common_edges_last_year':len(common),'newly_recorded_edges':len(new),
                       'not_recorded_after_last_year':len(absent),'coverage_of_prior_edges':len(common)/len(prevkey) if prevkey else np.nan,
                       'new_edges_not_assumed_zero_before':True})
        for category,keys in [('newly_recorded',new),('no_longer_recorded',absent)]:
            for kk in sorted(keys): details.append({'year':y,'category':category,**dict(zip(k,kk))})
    return pd.DataFrame(annual),pd.DataFrame(details)

if CFG.RUN_EDGE_COVERAGE_AUDIT:
    ec,ed=edge_coverage_audit(od_inter)
    PANELS['od_edge_coverage']=ec
    if not ed.empty:PANELS['od_edge_transitions']=ed
    ISSUES.append({'module':'ML-5.1','status':'SCIENTIFIC_LIMITATION',
       'reason':'Тест ограничен OD с точно наблюдаемой исторической парой: новые и пропавшие рёбра — отдельный аудит, неизвестные значения не нули',
       'required_fields':'полная матрица с семантикой отсутствия записи для прогноза новых/исчезающих рёбер'})
    print('OD coverage: наблюдаемые новые/пропадающие направления по годам',ec[['year','newly_recorded_edges','not_recorded_after_last_year']].to_dict('records'))


## 8. Предрегистрация прогноза до просмотра тестовых ошибок

Единица прогноза — идентифицированная пара регионов или организация, год происхождения информации `origin_year`, горизонт `h=1`, конечный `target_year=origin_year+h`. Будущие значения не становятся признаками. Сравнение моделей только на совпадающих тестовых ключах. **Значения временного теста не используются ни для выбора модели, ни для подбора гиперпараметров**.

`ROLLING_TIME`: предпоследний год метки — development, последний — окончательный test; train — раньше. Если длины ряда/тестовой выборки не хватает, возвращается `NOT_FEASIBLE`. Простая интерпретация иерархических/групповых результатов — прогнозная, не причинная. Значение предыдущего года входит в baseline и модель; `t+1` не входит в матрицу признаков.

In [ ]:
EXPERIMENT_PLAN={
 "ML-5.1":{"target":"registered_interregional_spo_OD_count_on_persistent_observed_edges","base":"last observed flow","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"origin×destination","hypothesis":"H5.1"},
 "ML-5.2":{"target":"approved_annual_enrolment","base":"last institution value","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"institution/branch","hypothesis":"H5.2"},
 "ML-5.3":{"target":"registered_interregional_outflow_in_observed_universe","base":"previous outflow","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"region","hypothesis":"H5.3"},
 "ML-5.4":{"target":"programme enrolment only if verified","base":"previous programme value","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"region×programme","hypothesis":"H5.4"},
 "ML-5.5":{"target":"measured employment/wage if time available","base":"prior cohort observations","stats":"Ridge","ml":"CatBoost","cutoff":"employment measurement date","unit":"organization/cohort","hypothesis":"H5.5"},
 "ML-5.6":{"target":"observed network centrality/edges","base":"previous annual graph","stats":"Ridge","ml":"CatBoost","cutoff":"origin_year","unit":"region×year","hypothesis":"H5.6"},
 "ML-5.7":{"target":"municipal migration rate or change","base":"E0 prior migration and demographics","stats":"E0 Ridge","ml":"E1 CatBoost","cutoff":"origin_year","unit":"municipality","hypothesis":"H5.7"},
}
PLAN_HASH=hashlib.sha256(json.dumps({"experiment_plan":EXPERIMENT_PLAN,"config":asdict(CFG)},sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()

def make_temporal_panel(df,keys,target,features=None,h=1):
    assert h>=1
    if df.empty:return pd.DataFrame()
    keys=list(keys); required=keys+["year",target]
    if not set(required).issubset(df):raise KeyError("Нет целей/ключей: "+str(set(required)-set(df.columns)))
    if df.duplicated(keys+["year"]).any():raise ValueError("many-to-many в целевой панели")
    original=df.copy();original.year=normyear(original.year)
    original=original.dropna(subset=required)
    original[target]=numeric(original[target]);original=original.dropna(subset=[target])
    # Доступен только год y<=origin_year-publication_lag, а не значение origin_year.
    current=original[keys+["year",target]+(features or [])].rename(columns={target:"prior_value","year":"information_cutoff"})
    current["origin_year"]=(current.information_cutoff+CFG.PUBLICATION_LAG_YEARS).astype(int)
    future=original[keys+["year",target]].rename(columns={target:"actual","year":"target_year"})
    current["target_year"]=(current.origin_year+h).astype(int)
    # exact year-to-year join: нет искусственного перехода через пропущенные периоды
    panel=current.merge(future,on=keys+["target_year"],how="inner",validate="one_to_one")
    panel["entity_id"]=panel[keys].astype(str).agg("|".join,axis=1)
    panel["horizon"]=h
    assert (panel.information_cutoff<=panel.origin_year-CFG.PUBLICATION_LAG_YEARS).all()
    panel["source_vintage"]=INDEX["created_at"]
    return panel

def enforce_time(df):
    assert not df.empty
    assert (df.target_year>df.origin_year).all()
    assert (df.information_cutoff<=df.origin_year).all()
    assert not df.duplicated(["entity_id","origin_year","target_year"]).any()
    assert np.isfinite(df.actual).all()
    return True

def forecast_splits(df):
    years=sorted(df.target_year.dropna().unique().tolist())
    if len(years)<3:return None,"Менее трёх годов с целевыми метками для train/development/test"
    ydev,ytest=years[-2],years[-1]
    tr=df[df.target_year<ydev].copy();dev=df[df.target_year==ydev].copy();test=df[df.target_year==ytest].copy()
    if len(tr)<CFG.MIN_TRAIN_ROWS or len(dev)<CFG.MIN_TEST_ROWS or len(test)<CFG.MIN_TEST_ROWS:
        return None,f"Мало train/dev/test: {len(tr)}/{len(dev)}/{len(test)}"
    assert tr.target_year.max()<dev.target_year.min()<test.target_year.min()
    assert tr.target_year.max()<=dev.origin_year.min(), "Утечка будущих train-меток на DEV"
    assert dev.target_year.max()<=test.origin_year.min(), "Утечка DEV-меток на TEST"
    return {"train":tr,"dev":dev,"test":test,"dev_year":int(ydev),"test_year":int(ytest)},None

def future_mutation_canary(df,keys,target):
    # изменение ещё не известного значения t+1 не должно менять X(t)
    if df.empty:return False
    sorted_df=df.sort_values(keys+["year"]).copy()
    a=make_temporal_panel(sorted_df,keys,target)
    if a.empty:return False
    b=sorted_df.copy(); maxyear=int(b.year.max()); b.loc[b.year==maxyear,target]=numeric(b.loc[b.year==maxyear,target])+123456
    c=make_temporal_panel(b,keys,target)
    join_keys=["entity_id","origin_year","target_year"]
    pair=a.merge(c,on=join_keys,suffixes=("_a","_b"),validate="one_to_one")
    mask=pair.target_year==maxyear
    if not mask.any():return False
    return bool(np.allclose(pair.loc[mask,"prior_value_a"],pair.loc[mask,"prior_value_b"],equal_nan=True))

# Data gate после подготовки 7 независимых предметных задач, без обращения к финальному test.
TASK_DATA={}
if len(od_inter) and od_inter.year.nunique()>=CFG.MIN_YEARS:
    TASK_DATA["ML-5.1"]=make_temporal_panel(od_inter,["origin_region","destination_region","sector"],"flow_count")
else:issue("ML-5.1","Недостаточно наблюдений OD для годового прогноза","origin_region,destination_region,year,flow_count")
if not org_year.empty and org_year.year.nunique()>=CFG.MIN_YEARS:
    TASK_DATA["ML-5.2"]=make_temporal_panel(org_year,["entity_id","sector"],"admissions",features=["group_id"])
else:issue("ML-5.2","Нет утверждённых точных показателей зачисления или недостаточная годовая панель","id, year, admission_count, institution/branch")
if len(od_inter):
    outbound=od_inter.groupby(["origin_region","sector","year"],as_index=False,dropna=False).flow_count.sum()
    outbound=outbound.rename(columns={"flow_count":"interregional_outflow_observed"})
    PANELS["outflow_interregional_observed"]=outbound
    if outbound.year.nunique()>=CFG.MIN_YEARS:
        TASK_DATA["ML-5.3"]=make_temporal_panel(outbound,["origin_region","sector"],"interregional_outflow_observed")
    else:issue("ML-5.3","Недостаточно лет OD; доля от всех абитуриентов недоступна","полный знаменатель для доли выбывших")
if program.empty or not CFG.APPROVED_SPO_ADMISSIONS:
    issue("ML-5.4","Не определено по исходному показателю, какие строки программ являются поступлением","точный indicator_name, crosswalk классификаторов")
else:
    sel=program.loc[program.indicator_name.isin(CFG.APPROVED_SPO_ADMISSIONS)].copy()
    if len(sel) and not sel.duplicated(["region_name","specialty_code","year"]).any() and sel.year.nunique()>=CFG.MIN_YEARS:
        TASK_DATA["ML-5.4"]=make_temporal_panel(sel.rename(columns={"value":"programme_admissions"}),["region_name","specialty_code","sector"],"programme_admissions")
    else:issue("ML-5.4","Нельзя однозначно трактовать программу или размерность исходных строк","programme×region×year, admission_count")
issue("ML-5.5","E04 фиксирует измерение занятости по состоянию на 2024-12-31; ряд независимых дат измерения пока не подтверждён. География работы отсутствует","employment_observation_year, employment_location для retention")
if od_inter.year.nunique()<CFG.NETWORK_MIN_YEARS:issue("ML-5.6","Нет нескольких достоверных годовых сетей","origin,destination,year,flow_count")
# ML-5.7 разрешается только после независимой верификации географии организаций, ниже.
print("Предрегистрация:",PLAN_HASH[:16],"· прогнозные панели:",{k:len(v) for k,v in TASK_DATA.items()})

## 8.1. Исторические факторы и реальный тест против утечек

Для каждого origin используется только информация не позже `information_cutoff`. Экономические факторы без сопоставимых наблюдений не выдумываются. Наблюдаемые маргинали и количество организаций СПО — только ретроспективные признаки, **не** характеристики будущего года.

In [ ]:
# Строго исторические маргинали и масштаб СПО из наблюдённых лет.
# Маргинали рассчитываются ТОЛЬКО из od_inter на information_cutoff, никогда в target_year.
def enrich_od_features(pairs,observed):
    if pairs.empty:return pairs.copy()
    x=pairs.copy()
    v=observed.loc[(~unknown_region(observed.origin_region))&(~unknown_region(observed.destination_region))&
                   (observed.origin_region!=observed.destination_region)].copy()
    if len(v)==0:return x
    origin=v.groupby(['year','origin_region'],as_index=False).agg(
        historical_origin_inter_flow=('flow_count','sum'),historical_origin_n_destinations=('destination_region','nunique'))
    dest=v.groupby(['year','destination_region'],as_index=False).agg(
        historical_destination_inter_flow=('flow_count','sum'),historical_destination_n_origins=('origin_region','nunique'))
    common=['information_cutoff']
    origin=origin.rename(columns={'year':'information_cutoff'})
    dest=dest.rename(columns={'year':'information_cutoff'})
    if 'origin_region' in x:
        x=x.merge(origin,on=['information_cutoff','origin_region'],how='left',validate='many_to_one')
    if 'destination_region' in x:
        x=x.merge(dest,on=['information_cutoff','destination_region'],how='left',validate='many_to_one')
    # Count of organizations is NOT a claim of available admission capacity.
    if 'destination_region' in x and 'spo_reg' in globals() and isinstance(spo_reg,pd.DataFrame) and len(spo_reg):
        src=spo_reg[['year','region_oktmo','entity_id']].dropna().drop_duplicates()
        src['region_oktmo']=normcode(src.region_oktmo)
        counts=src.groupby(['year','region_oktmo'],as_index=False).entity_id.nunique().rename(
            columns={'year':'information_cutoff','region_oktmo':'destination_region','entity_id':'historical_spo_orgs_destination'})
        x=x.merge(counts,on=['information_cutoff','destination_region'],how='left',validate='many_to_one')
    assert len(x)==len(pairs), 'JOIN изменил количество OD-строк'
    assert not x.duplicated(['entity_id','origin_year','target_year']).any()
    return x

# Второй лаг доступен не позже information_cutoff-1; без подглядывания в target.
def add_second_historical_lag(pairs, observations, keys, value):
    if pairs.empty:return pairs.copy()
    base=observations[list(keys)+['year',value]].copy().rename(columns={'year':'old_year',value:'prior2_value'})
    base['information_cutoff']=base.old_year.astype(int)+1
    base=base.drop(columns=['old_year'])
    if base.duplicated(list(keys)+['information_cutoff']).any():
        raise AssertionError('Дубли исторических ключей второго лага')
    enriched=pairs.merge(base,on=list(keys)+['information_cutoff'],how='left',validate='many_to_one')
    assert len(enriched)==len(pairs)
    return enriched

if 'ML-5.1' in TASK_DATA:
    TASK_DATA['ML-5.1']=add_second_historical_lag(enrich_od_features(TASK_DATA['ML-5.1'],od_inter),od_inter,['origin_region','destination_region','sector'],'flow_count')
if 'ML-5.3' in TASK_DATA:
    TASK_DATA['ML-5.3']=add_second_historical_lag(enrich_od_features(TASK_DATA['ML-5.3'],od_inter),PANELS['outflow_interregional_observed'],['origin_region','sector'],'interregional_outflow_observed')
PANELS['forecast_feature_coverage']=pd.DataFrame([
    {'module':k,'feature':c,'count_non_missing':int(v[c].notna().sum()),'share_non_missing':float(v[c].notna().mean())}
    for k,v in TASK_DATA.items() for c in v.columns
    if c.startswith('historical_')])

def raw_future_mutation_canary(observed,task,keys,target):
    # Rebuild ALL historical features after mutating the last year of raw OD.
    if observed.empty or not task:return False
    copy=observed.copy()
    latest=int(copy.year.max())
    mask=copy.year==latest
    assert mask.any()
    copy.loc[mask,'flow_count']=copy.loc[mask,'flow_count']+100000.
    if task=='ML-5.1':
        a=make_temporal_panel(observed,keys,target)
        b=make_temporal_panel(copy,keys,target)
    elif task=='ML-5.3':
        def out(z):
            return z.groupby(['origin_region','sector','year'],as_index=False).flow_count.sum().rename(columns={'flow_count':'interregional_outflow_observed'})
        a=make_temporal_panel(out(observed),keys,target)
        b=make_temporal_panel(out(copy),keys,target)
    else:return False
    a=enrich_od_features(a,observed); b=enrich_od_features(b,copy)
    if task=='ML-5.1':
        a=add_second_historical_lag(a,observed,keys,target)
        b=add_second_historical_lag(b,copy,keys,target)
    else:
        a=add_second_historical_lag(a,out(observed),keys,target)
        b=add_second_historical_lag(b,out(copy),keys,target)
    cols=[c for c in a.columns if c in b.columns and (c in ('prior_value','prior2_value') or c.startswith('historical_'))]
    identifier=['entity_id','origin_year','target_year']
    check=a.loc[a.target_year==latest,identifier+cols+['actual']].merge(
        b.loc[b.target_year==latest,identifier+cols+['actual']],on=identifier,validate='one_to_one',suffixes=('_before','_after'))
    if len(check)==0 or not (check.actual_before!=check.actual_after).any():return False
    return all(np.allclose(pd.to_numeric(check[c+'_before'],errors='coerce'),
                           pd.to_numeric(check[c+'_after'],errors='coerce'),equal_nan=True)
               for c in cols)

CANARY_RESULTS={}
for name,keys,target in [('ML-5.1',['origin_region','destination_region','sector'],'flow_count'),
                         ('ML-5.3',['origin_region','sector'],'interregional_outflow_observed')]:
    if name in TASK_DATA:
        result=raw_future_mutation_canary(od_inter,name,keys,target)
        CANARY_RESULTS[name]=bool(result)
        assert result, 'FAIL_FUTURE_MUTATION: '+name
PANELS['canary_tests']=pd.DataFrame([{'module':k,'future_mutation_canary':'PASS' if v else 'FAIL'} for k,v in CANARY_RESULTS.items()])
print('Canary: пересчитаны исходные признаки после изменения наблюдений последнего года',CANARY_RESULTS)


## 9. Прогнозные модели, пространственные тесты, независимые контрольные группы

Исторический baseline, регуляризованная линейная модель и CatBoost обучаются на **одинаковом** наборе обучающих наблюдений. Выбор CatBoost/Ridge выполняется на development, тестовые ошибки вычисляются после фиксации протокола. Категориальная кодировка обучается только на train. Стохастические повторения выполняются только для CatBoost, а не для детерминированного baseline и Ridge.

Дополнительные тесты `TERRITORY_ONLY`/`CROSSED` выполняются лишь при достаточном числе независимых групп, не смешивая филиалы одного вуза и исходную территорию с целевой.

In [ ]:
def feature_columns(df):
    nums=[c for c in ["prior_value","prior2_value","origin_year","prev_population","prev_migration_rate","number_vo_campuses","historical_origin_inter_flow","historical_destination_inter_flow","historical_origin_n_destinations","historical_destination_n_origins","historical_spo_orgs_destination"] if c in df.columns]
    cats=[c for c in ["origin_region","destination_region","sector","group_id","region","specialty_code"] if c in df.columns]
    return nums,cats

def model_inputs(data,train,seed,kind="ridge"):
    nums,cats=feature_columns(data)
    if not nums:raise ValueError("Нет известных на origin числовых признаков")
    X=data[nums+cats].copy()
    for col in nums:X[col]=numeric(X[col])
    for col in cats:X[col]=X[col].fillna("__MISSING__").astype(str)
    y=data["actual"].to_numpy(dtype=float)
    if kind=="ridge":
        transforms=[("n",Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("scaler",StandardScaler())]),nums)]
        if cats:transforms.append(("c",OneHotEncoder(handle_unknown="ignore"),cats))
        prep=ColumnTransformer(transforms,sparse_threshold=0.4)
        pipe=Pipeline([("prep",prep),("ridge",Ridge(alpha=100.0,solver="lsqr"))])
        tr_x=train[nums+cats].copy()
        for c in cats:tr_x[c]=tr_x[c].fillna("__MISSING__").astype(str)
        pipe.fit(tr_x,train.actual.to_numpy(float))
        if CFG.MODE=="FULL" and int(seed)==int(CFG.FINAL_SEEDS[0]) and ACTIVE_MODULE:
            import joblib
            fp=OUT/"derived"/(ACTIVE_MODULE.replace("-", "_").replace(".", "_")+"_ridge.joblib")
            joblib.dump(pipe,fp,compress=3);MODEL_FILES[str(fp.name)]=fp
        return pipe.predict(X),{"kind":"ridge","alpha":100.0}
    if kind=="gravity_ppml":
        from sklearn.linear_model import PoissonRegressor
        gn=[c for c in ["prior_value","historical_origin_inter_flow","historical_destination_inter_flow",
                         "historical_origin_n_destinations","historical_destination_n_origins"] if c in nums]
        gc=[c for c in ["origin_region","destination_region"] if c in cats]
        if len(gc)<2 or len(gn)<3:
            raise ValueError('PPML: нужны прошлые массы и оба региона; отсутствие дистанции документировано')
        prep=ColumnTransformer([
            ('num',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),gn),
            ('geo',OneHotEncoder(handle_unknown='ignore'),gc)],sparse_threshold=0.3)
        reg=Pipeline([('prep',prep),('ppml',PoissonRegressor(alpha=1.,max_iter=130,tol=1e-4))])
        train_x=train[gn+gc].copy(); test_x=data[gn+gc].copy()
        for f in gn:
            train_x[f]=np.log1p(pd.to_numeric(train_x[f],errors='coerce').clip(lower=0))
            test_x[f]=np.log1p(pd.to_numeric(test_x[f],errors='coerce').clip(lower=0))
        for f in gc:
            train_x[f]=train_x[f].fillna('__MISSING__').astype(str)
            test_x[f]=test_x[f].fillna('__MISSING__').astype(str)
        reg.fit(train_x,np.maximum(0,train.actual.to_numpy(float)))
        return reg.predict(test_x),{'kind':'PPML_with_origin_destination_effects','uses_distance':False,
                                     'covariates':gn+gc,'alpha':1.0}
    if kind=="catboost":
        train_x=train[nums+cats].copy();x=X.copy()
        for c in nums:
            v=float(pd.to_numeric(train_x[c],errors="coerce").median())
            if not np.isfinite(v):v=0.0
            train_x[c]=pd.to_numeric(train_x[c],errors="coerce").fillna(v)
            x[c]=pd.to_numeric(x[c],errors="coerce").fillna(v)
        for c in cats:
            train_x[c]=train_x[c].fillna("__MISSING__").astype(str);x[c]=x[c].fillna("__MISSING__").astype(str)
        clf=CatBoostRegressor(iterations=90 if CFG.MODE=="SMOKE" else 320,depth=5,learning_rate=0.055,loss_function="RMSE",random_seed=int(seed),verbose=False,thread_count=2,l2_leaf_reg=8.0)
        clf.fit(train_x,train.actual.to_numpy(float),cat_features=cats)
        if CFG.MODE=="FULL" and int(seed)==int(CFG.FINAL_SEEDS[0]) and ACTIVE_MODULE:
            fp=OUT/"derived"/(ACTIVE_MODULE.replace("-", "_").replace(".", "_")+"_catboost.cbm")
            clf.save_model(str(fp));MODEL_FILES[str(fp.name)]=fp
        out=clf.predict(x)
        return out,{"kind":"catboost","seed":int(seed),"iterations":clf.tree_count_,"features":nums+cats,"feature_importance":dict(zip(nums+cats,clf.feature_importances_.tolist()))}
    raise ValueError(kind)

def predict_using(train,test,model,seed=17):
    if model=="baseline":return test.prior_value.to_numpy(dtype=float),{"kind":"persistence_with_publication_lag"}
    if model=="rolling2":
        # Среднее двух действительно наблюдавшихся исторических лет; отсутствие второго = инерция.
        a=pd.to_numeric(test.prior_value,errors='coerce').to_numpy(dtype=float)
        b=pd.to_numeric(test.prior2_value,errors='coerce').to_numpy(dtype=float)
        return np.maximum(0,np.where(np.isfinite(b),(a+b)/2,a)),{'kind':'two_known_observations_average'}
    if model=="damped_trend":
        a=pd.to_numeric(test.prior_value,errors='coerce').to_numpy(dtype=float)
        b=pd.to_numeric(test.prior2_value,errors='coerce').to_numpy(dtype=float)
        return np.maximum(0,np.where(np.isfinite(b),a+0.35*(a-b),a)),{'kind':'predefined_damped_two_observation_trend','damping':0.35}
    pred,meta=model_inputs(test,train,seed,kind=model)
    return pred,meta

def err(y,yp):
    mask=np.isfinite(y)&np.isfinite(yp)
    y=np.asarray(y)[mask];yp=np.asarray(yp)[mask]
    if len(y)==0:return {"MAE":np.nan,"RMSE":np.nan,"WAPE":np.nan,"n":0}
    den=np.abs(y).sum()
    return {"MAE":float(np.mean(np.abs(y-yp))),"RMSE":float(np.sqrt(np.mean((y-yp)**2))),"WAPE":float(np.abs(y-yp).sum()/den) if den>0 else np.nan,"n":int(len(y))}

def paired_block_ci(df,group_col="bootstrap_group",rep=None,seed=2026):
    rep=rep or (50 if CFG.MODE=="SMOKE" else CFG.BOOTSTRAP_REPS)
    groups=df[group_col].astype(str).unique();rng=np.random.default_rng(seed)
    baseline=np.abs(df.actual-df.pred_baseline).to_numpy(float)
    proposed=np.abs(df.actual-df.pred_model).to_numpy(float)
    bygrp=[np.flatnonzero(df[group_col].astype(str).to_numpy()==g) for g in groups]
    if len(bygrp)<5:return {"delta_MAE":np.nan,"ci_low":np.nan,"ci_high":np.nan,"p_two_sided":np.nan,"groups":len(groups)}
    draws=[]
    for _ in range(rep):
        picks=rng.integers(0,len(groups),len(groups))
        ix=np.concatenate([bygrp[i] for i in picks])
        draws.append(float((proposed[ix]-baseline[ix]).mean()))
    arr=np.asarray(draws)
    return {"delta_MAE":float((proposed-baseline).mean()),"ci_low":float(np.quantile(arr,0.025)),"ci_high":float(np.quantile(arr,0.975)),"p_two_sided":float(min(1.0,2*min((np.count_nonzero(arr<=0)+1)/(len(arr)+1),(np.count_nonzero(arr>=0)+1)/(len(arr)+1)))),"groups":len(groups)}

def run_predictive_module(module,frame,cluster_col,nonnegative=True):
    global ACTIVE_MODULE
    ACTIVE_MODULE=module
    if frame.empty:issue(module,"Нет модельной панели");return
    enforce_time(frame)
    if module=='ML-5.1':
        assert (frame.origin_region!=frame.destination_region).all(), 'В ML-5.1 попала диагональ'
    packs,why=forecast_splits(frame)
    if packs is None:issue(module,why,"несколько лет train/dev/test");return
    if module in ("ML-5.1","ML-5.3"): assert CANARY_RESULTS.get(module,False), "Реальная канарейка будущих данных провалена"
    tr,dev,test=packs["train"],packs["dev"],packs["test"]
    # Development (отбор): ни одна оценка test здесь недоступна.
    pdev={};dev_scores={}
    models=["baseline","rolling2","damped_trend","ridge","catboost"]+(["gravity_ppml"] if module=="ML-5.1" else [])
    for model in models:
        seeds=[CFG.DEV_SEEDS[0]] if model!="catboost" or CFG.MODE=="SMOKE" else list(CFG.DEV_SEEDS)
        scores=[]
        for seed in seeds:
            raw,meta=predict_using(tr,dev,model,seed)
            pred=np.maximum(0,raw) if nonnegative else raw
            scores.append(err(dev.actual.to_numpy(float),pred)["MAE"])
        dev_scores[model]=float(np.mean(scores))
    # На тесте никогда не выбираем модель; равенство трактуется в пользу более простой.
    preference={'baseline':0,'rolling2':1,'damped_trend':2,'ridge':3,'gravity_ppml':4,'catboost':5}
    winner=min(dev_scores,key=lambda m:(round(dev_scores[m],10),preference[m]))
    fulltrain=pd.concat([tr,dev],ignore_index=True)
    seed_count=CFG.MAX_FINAL_SEEDS_SMOKE if CFG.MODE=="SMOKE" else (len(CFG.FINAL_SEEDS) if winner=="catboost" else 1)
    predictions={}; allmeta=[]
    for model in models:
        seeds=[CFG.FINAL_SEEDS[0]] if model!="catboost" else list(CFG.FINAL_SEEDS[:seed_count])
        for seed in seeds:
            raw,meta=predict_using(fulltrain,test,model,seed)
            pred=np.maximum(0,raw) if nonnegative else raw
            e=err(test.actual.to_numpy(float),pred)
            METRICS.append({"module":module,"protocol":"ROLLING_TIME","model":model,"seed":int(seed) if model=="catboost" else -1,"selected_on_dev":model==winner,"dev_mae_mean":dev_scores[model],"target_year":int(packs["test_year"]),**e})
            out=test[[c for c in test.columns if c in ["entity_id","origin_year","target_year","horizon","information_cutoff","source_vintage","origin_region","destination_region","sector","group_id","region"]]].copy()
            out["module"]=module;out["split_id"]="ROLLING_TIME";out["model"]=model;out["seed"]=int(seed) if model=="catboost" else -1
            out["actual"]=test.actual.to_numpy(float);out["predicted"]=pred
            predictions[(model,int(seed))]=out
            allmeta.append({"module":module,**meta})
    # Итоговый ансамбль предопределён до теста: среднее ВСЕХ заранее заданных сидов.
    if any(k[0]=='catboost' for k in predictions):
        cb=[v for (m,_),v in predictions.items() if m=='catboost']
        ensemble=cb[0].copy()
        ensemble['predicted']=np.mean(np.vstack([x.predicted.to_numpy(float) for x in cb]),axis=0)
        ensemble['model']='catboost_ensemble';ensemble['seed']=-1
        predictions[('catboost_ensemble',-1)]=ensemble
        eo=err(test.actual.to_numpy(float),ensemble.predicted.to_numpy(float))
        METRICS.append({'module':module,'protocol':'ROLLING_TIME','model':'catboost_ensemble','seed':-1,
                        'selected_on_dev':winner=='catboost','dev_mae_mean':dev_scores['catboost'],
                        'target_year':int(packs['test_year']),**eo})
    PREDICTIONS.extend(predictions.values());MODELS_META.extend(allmeta)
    # Paired uncertainty using common keys. Stochastic results average over their seeds for primary comparison.
    base=predictions[("baseline",int(CFG.FINAL_SEEDS[0]))]
    for model in [m for m in models if m!="baseline"]:
        compared=[x for (m,s),x in predictions.items() if m==model]
        p=np.mean(np.vstack([x.predicted.to_numpy(float) for x in compared]),axis=0)
        comp=pd.DataFrame({"actual":base.actual.to_numpy(float),"pred_baseline":base.predicted.to_numpy(float),"pred_model":p,
                        "bootstrap_group":test[cluster_col].astype(str).to_numpy()})
        ci=paired_block_ci(comp)
        METRICS.append({"module":module,"protocol":"BLOCK_BOOTSTRAP","model":model,"seed":-1,"selected_on_dev":model==winner,"dev_mae_mean":dev_scores[model],"target_year":int(packs["test_year"]),"MAE":err(comp.actual,p)["MAE"],"delta_MAE":ci["delta_MAE"],"ci_low":ci["ci_low"],"ci_high":ci["ci_high"],"p_two_sided":ci["p_two_sided"],"clusters":ci["groups"],"n":len(comp)})
    baseline_mae=err(test.actual.to_numpy(float),predictions[('baseline',int(CFG.FINAL_SEEDS[0]))].predicted.to_numpy(float))['MAE']
    chosen_model='catboost_ensemble' if winner=='catboost' else winner
    chosen_key=(chosen_model,-1) if chosen_model=='catboost_ensemble' else (chosen_model,int(CFG.FINAL_SEEDS[0]))
    chosen_mae=err(test.actual.to_numpy(float),predictions[chosen_key].predicted.to_numpy(float))['MAE']
    PANELS.setdefault('confirmatory_results',[]).append({'module':module,'selected_dev_model':winner,'selected_test_mae':chosen_mae,
       'baseline_test_mae':baseline_mae,'test_delta_mae':chosen_mae-baseline_mae,
       'improvement_status':'IMPROVED' if chosen_mae<baseline_mae else 'NO_IMPROVEMENT',
       'test_year':int(packs['test_year']),'common_test_keys':len(test)})
    SPLITS.append({"module":module,"train":len(tr),"dev":len(dev),"test":len(test),"dev_year":packs["dev_year"],"test_year":packs["test_year"],"chosen_dev_model":winner,"group_holdout":"см. отдельную проверку"})
    STATUS[module]={"status":"TESTED_SMOKE" if CFG.MODE=="SMOKE" else "TESTED_FULL",
       "reason":f"Межрегиональный поток без диагонали; test {packs['test_year']}; DEV-selected: {winner}; test delta_MAE={chosen_mae-baseline_mae:+.4f}. "
                "Отсутствующие рёбра не достраиваются; H5.1 не идентифицирует экономические причины"}
    return packs


# Задача 5.7: E0 и E1 на общей муниципальной поддержке, без искусственного отсутствия вузов.
MUNI_E0_E1=None
if CFG.MODE=="FULL" and not vo_reg.empty and not mun.empty:
    known=vo_reg.dropna(subset=["oktmo","year","entity_id"])
    conf=known.groupby(["oktmo","year"],as_index=False).agg(number_vo_campuses=("entity_id","nunique"),organisations=("entity_id",lambda s:",".join(sorted(set(s.astype(str).tolist()))[:5])))
    # Предыдущий год, а не будущий: E1 не получает информацию об открытии после origin.
    lagged=conf.rename(columns={"year":"edu_year"})
    base=mun[["oktmo","region","year","migration_rate","population"]].dropna(subset=["migration_rate","population"]).copy()
    base=base.rename(columns={"year":"available_year","migration_rate":"prev_migration_rate","population":"prev_population"})
    base["origin_year"]=base.available_year+CFG.PUBLICATION_LAG_YEARS
    base["target_year"]=base.origin_year+1
    later=mun[["oktmo","year","migration_rate"]].rename(columns={"year":"target_year","migration_rate":"actual"})
    base=base.merge(later,on=["oktmo","target_year"],validate="one_to_one")
    base["edu_year"]=base.available_year
    base=base.merge(lagged,on=["oktmo","edu_year"],how="inner",validate="many_to_one")
    base["entity_id"]=base.oktmo.astype(str);base["horizon"]=1
    base["prior_value"]=base.prev_migration_rate
    base["information_cutoff"]=base.available_year
    base["source_vintage"]=INDEX["created_at"]
    ISSUES.append({"module":"ML-5.7","status":"SCIENTIFIC_LIMITATION","reason":"E1 содержит только площадки ВО с точным ОКТМО. Для полного ВО+СПО необходим подтверждённый адрес площадки СПО, совпадающий с ОКТМО муниципалитета","required_fields":"verified SPO municipality campus location"})
    PANELS["municipality_education_context"]=base
    gate("municipality_education_context",base,["entity_id","origin_year","target_year"])
    if base.entity_id.nunique()>=CFG.MIN_MUNI_E1_SUPPORT and base.target_year.nunique()>=3 and base.number_vo_campuses.nunique()>1:
        MUNI_E0_E1=base
    else:issue("ML-5.7",f"Мало точных ОКТМО совпадений или вариации инфраструктуры: n={base.entity_id.nunique()}, years={base.target_year.nunique()}","validated campus OKTMO, stable municipality period, migration_rate")
else:issue("ML-5.7","Требуются идентифицированные площадки ВО/СПО с точным муниципальным ОКТМО","municipal campus location; matching observations")

# Реальные модельные задачи. Прогресс-бар создаётся один раз после планирования.
MODEL_JOBS=[(k,v) for k,v in TASK_DATA.items() if not v.empty and forecast_splits(v)[0] is not None]

for k,v in TASK_DATA.items():
    if k not in [a for a,_ in MODEL_JOBS] and STATUS[k]["status"]=="PENDING":issue(k,forecast_splits(v)[1] or "недостаточная выборка","train/dev/test")
if MUNI_E0_E1 is not None and forecast_splits(MUNI_E0_E1)[0] is None:issue("ML-5.7","Мало лет/наблюдений на общей поддержке E0/E1","train/dev/test")

# Профиль SMOKE исполняет только доступную задачу OD, без научной публикации.
if CFG.MODE=="SMOKE":MODEL_JOBS=[x for x in MODEL_JOBS if x[0]=="ML-5.1"][:1]
BAR=tqdm(total=len(MODEL_JOBS)+7,desc="ML-05 · модель → QA → публикация",unit="этап",leave=True)
for mod,frame in MODEL_JOBS:
    BAR.set_postfix_str(mod)
    cluster="origin_region" if "origin_region" in frame else ("region" if mod=="ML-5.7" else ("group_id" if "group_id" in frame else "entity_id"))
    try:run_predictive_module(mod,frame,cluster,nonnegative=(mod!="ML-5.7"))
    except Exception as e:
        STATUS[mod]={"status":"FAIL","reason":type(e).__name__+": "+str(e)[:180]}
        (OUT/"traceback.log").write_text(traceback.format_exc(),encoding="utf-8")
        if CFG.MODE=="FULL":raise
    BAR.update(1)
print("Фактические модели:",len(METRICS),"строк оценок;",len(PREDICTIONS),"пакетов предсказаний")

## 10. Модуль ML-5.6: структура и устойчивость наблюдаемых образовательных сетей

Направленные рёбра строятся только из сохранившихся фактических OD-строк. Силы, PageRank и сообщества рассчитаны по каждой годовой сети; наличие нулевых/ненаблюдаемых рёбер не подменяется. Различие классификаторов или определений региона требует отдельного сопоставления.

In [ ]:
NETWORK=[];NET_EDGES=[];NET_SETS={}
if not od_inter.empty and od_inter.year.nunique()>=CFG.NETWORK_MIN_YEARS:
    for yr,sub in od_inter.groupby("year"):
        G=nx.DiGraph()
        for _,row in sub.iterrows():
            if row.flow_count>0:G.add_edge(str(row.origin_region),str(row.destination_region),weight=float(row.flow_count))
        assert nx.number_of_selfloops(G)==0, 'Межрегиональный граф содержит петлю'
        if not G.number_of_edges():continue
        ins=dict(G.in_degree(weight="weight"));outs=dict(G.out_degree(weight="weight"))
        pager=nx.pagerank(G,weight="weight",alpha=0.85,max_iter=300)
        comm=list(nx.algorithms.community.louvain_communities(G.to_undirected(),weight="weight",seed=173))
        commap={node:i for i,part in enumerate(comm) for node in part}
        NET_SETS[int(yr)]={frozenset(c) for c in comm}
        total=sum(float(d["weight"]) for _,_,d in G.edges(data=True))
        for node in G.nodes():NETWORK.append({"year":int(yr),"region_code":node,"inflow_observed":ins.get(node,0.),"outflow_observed":outs.get(node,0.),"pagerank":pager.get(node,0.),"community":commap.get(node,-1),"total_observed_flow":total,"edges":G.number_of_edges(),"diagonal_in_graph":0})
        for u,v,d in G.edges(data=True):NET_EDGES.append({"year":int(yr),"origin_region":u,"destination_region":v,"flow_count":d["weight"]})
    nodes=pd.DataFrame(NETWORK)
    PANELS["edu_network_year"]=nodes
    PANELS["edu_network_edges"]=pd.DataFrame(NET_EDGES)
    gate("edu_network_year",nodes,["region_code","year"])
    if nodes.year.nunique()>=2:
        years=sorted(NET_SETS)
        jacc=[]
        for a,b in zip(years[:-1],years[1:]):
            p,q=NET_SETS[a],NET_SETS[b]
            common=set.union(*(map(set,p))) & set.union(*(map(set,q))) if p and q else set()
            if common:
                pairs_a={tuple(sorted((u,v))) for part in p for u in part for v in part if u<v and u in common and v in common}
                pairs_b={tuple(sorted((u,v))) for part in q for u in part for v in part if u<v and u in common and v in common}
                denom=len(pairs_a|pairs_b)
                jacc.append({"year_from":a,"year_to":b,"pairwise_community_jaccard":len(pairs_a&pairs_b)/denom if denom else np.nan,"common_regions":len(common)})
        PANELS["network_stability"]=pd.DataFrame(jacc)
        STATUS["ML-5.6"]={"status":"DESCRIPTIVE_PASS","reason":f"Наблюдаемые межрегиональные сети без петель: {len(years)} лет; проверены сообщества и устойчивость"}
    else:issue("ML-5.6","Недостаточно независимых годов для устойчивости сетей")
else:issue("ML-5.6","Недостаточно реальных OD для наблюдаемой пространственной сети")
BAR.update(1)

## 11. Групповые и пространственные тесты, статистическая неопределённость

На реальных тестовых ключах применяется блочный bootstrap по региону происхождения или по родительской образовательной организации (как задокументировано). Проверяется чувствительность к отбору крупнейших потоков. Семь гипотез не агрегируются в один score. Для заранее заявленных семи независимых сравнений применяется Holm; интерпретация остаётся исследовательской даже при положительной метрике.

In [ ]:
# Group/CROSSED на исходных данных только для достаточных групп; тест независим от временного test.
def group_protocol(module,df,group_col,kind='CROSSED'):
    groups=sorted(df[group_col].astype(str).unique())
    if len(groups)<12:return {'module':module,'protocol':kind,'status':'NOT_FEASIBLE','reason':'Мало независимых групп'}
    hashed={v:int(hashlib.sha256((module+'|'+v).encode()).hexdigest()[:8],16)%5==0 for v in groups}
    mask=df[group_col].astype(str).map(hashed)
    lastyear=int(df.target_year.max())
    if kind=='CROSSED':
        train=df[(~mask)&(df.target_year<lastyear)].copy()
        test=df[(mask)&(df.target_year==lastyear)].copy()
    else:
        train=df[(~mask)&(df.target_year<lastyear)].copy()
        test=df[(mask)&(df.target_year<lastyear)].copy()
    if len(train)<CFG.MIN_TRAIN_ROWS or len(test)<CFG.MIN_TEST_ROWS:
        return {'module':module,'protocol':kind,'status':'NOT_FEASIBLE','reason':'Мало отложенных ключей'}
    assert set(train[group_col].astype(str)).isdisjoint(set(test[group_col].astype(str)))
    b,_=predict_using(train,test,'baseline')
    winner=next((x['chosen_dev_model'] for x in SPLITS if x['module']==module),None)
    if winner not in ('baseline','rolling2','damped_trend','ridge','catboost','gravity_ppml'):winner='baseline'
    m,_=predict_using(train,test,winner,CFG.DEV_SEEDS[0])
    m=np.maximum(0,m)
    eb,em=err(test.actual,b),err(test.actual,m)
    return {'module':module,'protocol':kind,'status':'TESTED','train':len(train),'test':len(test),
            'group_count_train':train[group_col].nunique(),'group_count_test':test[group_col].nunique(),
            'comparison_model':winner,'baseline_MAE':eb['MAE'],'comparison_MAE':em['MAE'],
            'delta_MAE':em['MAE']-eb['MAE'],'test_year':lastyear if kind=='CROSSED' else 'historical'}

GROUP_PROTOCOLS=[]
if CFG.MODE=="FULL":
    for mod,frame in TASK_DATA.items():
        if len(frame):
            col="origin_region" if "origin_region" in frame else "group_id" if "group_id" in frame else None
            if col:
                GROUP_PROTOCOLS.append(group_protocol(mod,frame,col,'CROSSED'))
                GROUP_PROTOCOLS.append(group_protocol(mod,frame,col,'TERRITORY_ONLY'))
PANELS["group_protocols"]=pd.DataFrame(GROUP_PROTOCOLS)

# H5.7: два одинаковых алгоритма Ridge, различаются только в доступности образовательного признака.
def run_e0_e1(base):
    pack,why=forecast_splits(base)
    if pack is None:issue("ML-5.7",why);return
    train=pd.concat([pack["train"],pack["dev"]],ignore_index=True)
    test=pack["test"].copy()
    covars0=["prior_value","prev_population","origin_year"]
    covars1=covars0+["number_vo_campuses"]
    estimators={}
    for name,cols in [("E0",covars0),("E1",covars1)]:
        x=train[cols].apply(pd.to_numeric,errors="coerce")
        xt=test[cols].apply(pd.to_numeric,errors="coerce")
        model=Pipeline([("impute",SimpleImputer(strategy="median",add_indicator=True)),("scale",StandardScaler()),("ridge",Ridge(alpha=100.0))])
        model.fit(x,train.actual.astype(float))
        yp=model.predict(xt);estimators[name]=yp
        e=err(test.actual,yp)
        METRICS.append({"module":"ML-5.7","protocol":"ROLLING_TIME_COMMON_SUPPORT","model":name,"seed":-1,"target_year":int(pack["test_year"]),**e})
        p=test[["entity_id","origin_year","target_year","horizon","information_cutoff","source_vintage","region"]].copy()
        p["actual"]=test.actual.astype(float);p["predicted"]=yp;p["module"]="ML-5.7";p["model"]=name;p["seed"]=-1;p["split_id"]="E0_E1_common_support"
        PREDICTIONS.append(p)
    comp=pd.DataFrame({"actual":test.actual.to_numpy(float),"pred_baseline":estimators["E0"],"pred_model":estimators["E1"],"bootstrap_group":test.region.astype(str)})
    delta=paired_block_ci(comp)
    METRICS.append({"module":"ML-5.7","protocol":"BLOCK_BOOTSTRAP","model":"E1_minus_E0","seed":-1,"target_year":int(pack["test_year"]),"n":len(test),**delta})
    SPLITS.append({"module":"ML-5.7","train":len(pack["train"]),"dev":len(pack["dev"]),"test":len(test),"dev_year":pack["dev_year"],"test_year":pack["test_year"],"chosen_dev_model":"not_selected; fixed E0/E1"})
    STATUS["ML-5.7"]={"status":"TESTED_FULL","reason":"E0/E1 проверены на точном общем множестве ОКТМО и годовых меток; отсутствие СПО не приписывается"}

if CFG.MODE=="FULL" and MUNI_E0_E1 is not None:run_e0_e1(MUNI_E0_E1)

# Препроцессинг задним числом не гарантирует доступность на дату прогноза: фиксируем ограничение.
ISSUES.append({"module":"ALL","status":"SCIENTIFIC_LIMITATION","reason":"Нет пообъектных available_at: историческая валидация при консервативном publication lag; не выдавать за доказанную реалистичную vintage-репликацию","required_fields":"publication_year / available_at per field"})

metrics_df=pd.DataFrame(METRICS)
if not metrics_df.empty and "p_two_sided" in metrics_df:
    confirm=metrics_df.loc[(metrics_df.protocol=="BLOCK_BOOTSTRAP") & metrics_df.p_two_sided.notna()].copy()
    confirm["holm_p"]=np.nan
    if len(confirm):
        p=np.asarray(confirm.p_two_sided,dtype=float);idx=np.argsort(p);adj=np.empty_like(p)
        for rank,j in enumerate(idx):adj[j]=min(1.0,p[j]*(len(p)-rank))
        adj[idx]=np.maximum.accumulate(adj[idx]);confirm["holm_p"]=adj
        metrics_df=metrics_df.merge(confirm[["module","model","protocol","holm_p"]],on=["module","model","protocol"],how="left",validate="many_to_one")
PANELS["metrics"]=metrics_df
print("Статусы по семи модулям:",{k:v["status"] for k,v in STATUS.items()})
BAR.update(1)

## 12. Визуализация реальных данных и интерпретация

Рисунки имеют единицы измерения, период и принадлежность первоисточнику. При отсутствии подтверждённого показателя не создаётся декоративная «карта» или пустая диаграмма. Отклонение прогноза указывает на статистическую ошибку, а не на эффект вмешательства в управление приёмом.

In [ ]:
def plot_save(name):
    fp=OUT/"figures"/(name+".png")
    plt.tight_layout();plt.savefig(fp,dpi=150,bbox_inches="tight");plt.close()
    FIGURES.append(str(fp));return fp

# Разделить неизвестные, диагональ и межрегиональные переходы также на рисунках.
if not od_inter.empty:
    yr=int(od_inter.year.max());e=od_inter.loc[od_inter.year==yr].copy()
    names=od[['destination_region','destination_region_name']].dropna().drop_duplicates()
    names=names.loc[~names.duplicated('destination_region',keep=False)]
    names_map=dict(zip(names.destination_region.astype(str),names.destination_region_name.astype(str)))
    def display_region(code):return names_map.get(str(code),str(code))
    top=e.groupby('destination_region',as_index=False).flow_count.sum().nlargest(15,'flow_count')
    fig,ax=plt.subplots(figsize=(12,6))
    ax.barh([display_region(c) for c in top.destination_region.astype(str)][::-1],top.flow_count.iloc[::-1],color='#2374A8')
    ax.set_xlabel('Наблюдаемые поступления из других регионов, чел.')
    ax.set_title(f'СПО: межрегиональные направления поступления, {yr} (без диагонали)')
    plot_save('01_interregional_spo_destination_top15')
    origins=e.groupby('origin_region').flow_count.sum().nlargest(14).index
    dests=e.groupby('destination_region').flow_count.sum().nlargest(14).index
    mat=e.loc[e.origin_region.isin(origins)&e.destination_region.isin(dests)].pivot_table(
        index='origin_region',columns='destination_region',values='flow_count',aggfunc='sum')
    # Отсутствующие пары не нули. NaN маскируется, а нулевые ячейки остаются нулевыми.
    vals=np.ma.masked_invalid(mat.to_numpy(dtype=float))
    cmap=plt.get_cmap('viridis').copy();cmap.set_bad('#e7e7e7')
    fig,ax=plt.subplots(figsize=(11,8))
    im=ax.imshow(vals,aspect='auto',cmap=cmap)
    ax.set_xticks(range(len(mat.columns)),[display_region(c) for c in mat.columns],rotation=90,fontsize=8)
    ax.set_yticks(range(len(mat.index)),[display_region(c) for c in mat.index],fontsize=8)
    ax.set_title(f'СПО: зарегистрированные межрегиональные OD, {yr}; серый = нет записи')
    fig.colorbar(im,ax=ax,label='Наблюдаемое число зачисленных, чел.')
    plot_save('02_interregional_od_heatmap')

if PREDICTIONS:
    pred=pd.concat(PREDICTIONS,ignore_index=True)
    chosen=next((v['chosen_dev_model'] for v in SPLITS if v['module']=='ML-5.1'),None)
    chosen='catboost_ensemble' if chosen=='catboost' else chosen
    if chosen is not None:
        p=pred.loc[(pred.module=='ML-5.1')&(pred.model==chosen)].copy()
        if len(p):
            p=p.nlargest(min(350,len(p)),'actual')
            fig,ax=plt.subplots(figsize=(7.2,6))
            ax.scatter(p.actual,p.predicted,s=13,alpha=.55,color='#116D9B',label=chosen)
            lim=max(float(p.actual.max()),float(p.predicted.max()),1)
            ax.plot([0,lim],[0,lim],'--',linewidth=1.3,color='#D35B4F',label='Точный прогноз')
            ax.set_xlabel('Факт: межрегиональные OD, чел.')
            ax.set_ylabel('Прогноз '+chosen+', чел.')
            ax.set_title('СПО: тестовые межрегиональные направления · без диагонали')
            ax.legend(loc='upper left');plot_save('03_interregional_od_test_prediction')
    p=pred.loc[(pred.module=='ML-5.7')]
    if len(p):
        sums=p.assign(abs_error=(p.actual-p.predicted).abs()).groupby('model').abs_error.mean()
        fig,ax=plt.subplots(figsize=(6,4));ax.bar(sums.index,sums.values)
        ax.set_ylabel('MAE, миграционный коэффициент муниципалитета')
        ax.set_title('E0 против E1: одинаковая тестовая поддержка')
        plot_save('04_municipal_e0_e1')
BAR.update(1)

# Интерпретация строится только по фактическим сравнениям.
INTERPRETATION=[]
for mod in STATUS:
    sub=metrics_df.loc[metrics_df.module==mod] if not metrics_df.empty else pd.DataFrame()
    if sub.empty:
        INTERPRETATION.append({"module":mod,"conclusion":"Независимое сравнение моделей отсутствует; " + STATUS[mod]["reason"],"evidence":"DATA_GATE/статус"})
        continue
    win=sub.loc[sub.protocol=='ROLLING_TIME']
    if len(win):
        b=win.loc[win.model=='baseline']
        selected=next((x['chosen_dev_model'] for x in SPLITS if x['module']==mod),None)
        if selected=='catboost': selected='catboost_ensemble'
        chosen=win.loc[win.model==selected]
        if not b.empty and not chosen.empty:
            baseline_mae=float(b.iloc[0].MAE);sel_mae=float(chosen.iloc[0].MAE)
            msg=(f'Предварительно отобранная на DEV модель {selected}: test MAE={sel_mae:.4g}; '
                 f'инерция={baseline_mae:.4g}. Диагональ и неизвестные регионы исключены. '
                 f'Разность MAE={sel_mae-baseline_mae:+.4g}; улучшение='+('ДА. ' if sel_mae<baseline_mae else 'НЕТ. ')+
                 'Нет подтверждённого места проживания и полной проверки экономических факторов.')
        else:msg='Для подтверждения нет сопоставимого DEV-selected и baseline'
    else:msg='Нет независимой прогнозной оценки; см. статус DATA_GATE'

    INTERPRETATION.append({"module":mod,"conclusion":msg,"evidence":"метрики + прогнозы в evidence"})
PANELS["interpretation"]=pd.DataFrame(INTERPRETATION)
BAR.update(1)
# Holm по заранее заданным двум подтверждающим сравнениям ML-5.1 и ML-5.3.
# Применяется к парному блочному bootstrap DEV-selected модели против инерции.
confirm=[]
for row in PANELS.get('confirmatory_results',[]):
    subset=[r for r in METRICS if r.get('module')==row['module'] and r.get('protocol')=='BLOCK_BOOTSTRAP'
            and r.get('model')==row['selected_dev_model']]
    if subset and np.isfinite(subset[-1].get('p_two_sided',np.nan)):
        confirm.append({'module':row['module'],'model':row['selected_dev_model'],
                        'p_raw':float(subset[-1]['p_two_sided']),
                        'delta_MAE':row['test_delta_mae'],
                        'direction':'improvement' if row['test_delta_mae']<0 else 'deterioration'})
    elif row['selected_dev_model']=='baseline':
        confirm.append({'module':row['module'],'model':'baseline','p_raw':np.nan,'delta_MAE':0.,
                        'direction':'baseline_selected','p_holm':np.nan,'reject_at_005':False})
if confirm:
    order=sorted([i for i in range(len(confirm)) if np.isfinite(confirm[i]['p_raw'])],key=lambda i:confirm[i]['p_raw'])
    prev=0.0
    for rank,i in enumerate(order):
        prev=max(prev,min(1.,(len(order)-rank)*confirm[i]['p_raw']))
        confirm[i]['p_holm']=prev
        confirm[i]['reject_at_005']=bool(prev<0.05)
    PANELS['holm_confirmatory']=pd.DataFrame(confirm)

# Подтверждающее сравнение на общем тесте, без позитивной селекции.
if isinstance(PANELS.get('confirmatory_results'),list):
    PANELS['confirmatory_results']=pd.DataFrame(PANELS['confirmatory_results'])
# Дополнительные факты об ограничениях, даже если модель относительно baseline улучшилась.
ISSUES.append({'module':'ML-5.1/3','status':'SCIENTIFIC_LIMITATION',
   'reason':'Ограниченный universe: пара должна наблюдаться в cutoff и target year; новые и исчезнувшие OD не включены в первичную MAE',
   'required_fields':'полнота матрицы / различение истинного нуля и отсутствия'})


## 13. Экспорт: Excel, технический ZIP и доказательные таблицы

Формируются ровно четыре логических результата: сам ноутбук, один Excel, один ZIP и `run_manifest.json`. Полные панели и построчные прогнозы сохраняются в Parquet внутри ZIP. Excel содержит реальные компактные таблицы/метрики; недоступные исследования перечисляются на листе ограничений, без пустых заготовок.

In [ ]:
import zipfile, tempfile
EXCEL=OUT/"ML5_EDUCATION_MIGRATION_RESULTS.xlsx"
EVIDENCE=OUT/"ML5_EVIDENCE.zip"
MANIFEST=OUT/"run_manifest.json"
NOTEBOOK=OUT/"ML5_EDUCATION_MIGRATION.ipynb"

def sheet_frame(df,book,title,max_rows=20000):
    if not isinstance(df,pd.DataFrame) or df.empty:return
    name=title[:31];safe=df.head(max_rows).copy()
    for c in safe.columns:
        if pd.api.types.is_datetime64_any_dtype(safe[c]):safe[c]=safe[c].astype(str)
        if pd.api.types.is_object_dtype(safe[c]):
            safe[c]=safe[c].map(lambda x:json.dumps(x,ensure_ascii=False,default=str) if isinstance(x,(dict,list,tuple,set)) else str(x) if x is not None else "")
    safe.to_excel(book,sheet_name=name,index=False)
    ws=book.sheets[name];ws.freeze_panes(1,0)
    ws.autofilter(0,0,max(0,len(safe)),len(safe.columns)-1)
    fmt=book.book.add_format({"bold":True,"font_color":"#FFFFFF","bg_color":"#253C58","text_wrap":True,"valign":"vcenter","border":0})
    ws.set_row(0,30,fmt)
    for i,c in enumerate(safe.columns):
        sample=safe[c].astype(str).head(200)
        width=min(41,max(12,len(str(c))+2,int(sample.str.len().quantile(.85))+2 if len(sample) else 12))
        ws.set_column(i,i,width)
    ws.set_tab_color("#2A6F8D")

rows_status=[{"module":k,**v,"hypothesis":"H"+k.split("-")[1]} for k,v in STATUS.items()]
passport=pd.DataFrame([
    {"Параметр":"Исследование","Значение":"ML-05 Образовательная миграция"},
    {"Параметр":"Режим","Значение":CFG.MODE},
    {"Параметр":"Версия","Значение":CFG.VERSION},
    {"Параметр":"Приватный Stage I","Значение":CFG.REPO},
    {"Параметр":"Каталог","Значение":CFG.INDEX},
    {"Параметр":"Источники","Значение":str(len(ASSETS))},
    {"Параметр":"Подтверждённые СПО OD","Значение":str(len(od_inter))},
    {"Параметр":"Ограничение OD","Значение":"Только известные разные origin/destination; diagonal и unknown в отдельных панелях. Семантика E01 требует сверки"},
    {"Параметр":"Диагональ исключена","Значение":str(len(od_intra))},
    {"Параметр":"Регион неизвестен","Значение":str(len(od_unknown))},
    {"Параметр":"Защита информации","Значение":"Ненаблюдаемые связи, география трудоустройства и нули не реконструировались"},
    {"Параметр":"Состояние публикации","Значение":"SMOKE — запрещена; FULL — только после локального QA"}
])
issues_df=pd.DataFrame(ISSUES)

with pd.ExcelWriter(EXCEL,engine="xlsxwriter",engine_kwargs={"options":{"strings_to_formulas":False,"strings_to_urls":False}}) as writer:
    sheet_frame(passport,writer,"Паспорт")
    sheet_frame(pd.DataFrame(rows_status),writer,"Семь_модулей")
    sheet_frame(ASSET_AUDIT,writer,"Источники_ВО_СПО")
    sheet_frame(pd.DataFrame(QA),writer,"DATA_GATE")
    sheet_frame(pd.DataFrame(PROVENANCE),writer,"Происхождение_данных")
    sheet_frame(issues_df,writer,"Пределы_данных")
    sheet_frame(pd.DataFrame(INTERPRETATION),writer,"Семь_гипотез")
    sheet_frame(metrics_df,writer,"Метрики_и_CI")
    sheet_frame(pd.DataFrame(SPLITS),writer,"Разбиения")
    sheet_frame(PANELS.get("group_protocols"),writer,"Пространственный_тест")
    sheet_frame(PANELS.get('confirmatory_results'),writer,'Итоговая_оценка')
    sheet_frame(PANELS.get('holm_confirmatory'),writer,'Коррекция_Holm')
    sheet_frame(PANELS.get('od_edge_coverage'),writer,'Покрытие_OD_по_годам')
    sheet_frame(PANELS.get('od_edge_transitions'),writer,'Новые_и_пропавшие_OD',20000)
    sheet_frame(PANELS.get("edu_org_registry"),writer,"Организации_филиалы",4000)
    sheet_frame(PANELS.get("edu_org_year"),writer,"ВО_СПО_прием",10000)
    sheet_frame(PANELS.get("od_composition"),writer,"Состав_OD")
    sheet_frame(PANELS.get("edu_od_interregional"),writer,"OD_межрегиональные",10000)
    sheet_frame(PANELS.get("edu_od_intraregional"),writer,"OD_диагональ",1000)
    sheet_frame(PANELS.get("edu_od_unknown_region"),writer,"OD_неопределённые",1000)
    sheet_frame(PANELS.get("canary_tests"),writer,"Канарейка_утечки")
    sheet_frame(PANELS.get("forecast_feature_coverage"),writer,"Покрытие_факторов")
    sheet_frame(PANELS.get("outflow_interregional_observed"),writer,"Отток_межрегиональный",10000)
    sheet_frame(PANELS.get("edu_program_year"),writer,"Программы",10000)
    sheet_frame(PANELS.get("graduate_outcomes"),writer,"Выпускники_занятость",10000)
    sheet_frame(PANELS.get("edu_network_year"),writer,"Сеть_регионы")
    sheet_frame(PANELS.get("network_stability"),writer,"Сеть_по_годам")
    sheet_frame(PANELS.get("municipality_education_context"),writer,"Муниципалитеты_E0_E1",10000)
    sheet_frame(PANELS.get("spo_indicator_dictionary"),writer,"Словарь_СПО",2000)
    sheet_frame(PANELS.get("e02_indicator_dictionary"),writer,"Словарь_ВО",2000)

METHODS="""# ML-05. Методы, единицы наблюдения и ограничения

Происхождение данных: handoff/07_v2/dataset_index.json; все SHA записаны в sources.json.
OD: СПО, только пары origin != destination с известными кодами; диагональ и неизвестные регионы отдельно.
Отсутствующие пары и нулевой поток различаются. Отдельный аудит новых и исчезающих OD. OD по previous education не равно постоянной миграции.
Гравитационный PPML baseline использует лагированные массы и эффекты origin/destination; достоверных расстояний нет.
Future-mutation canary изменяет последние сырые OD и полностью перестраивает все модельные признаки.
Для количественных целей MAE и RMSE. WAPE агрегируется с защитой делителя от нуля.
Финальный вывод основан на отборе только по DEV. CatBoost оценивается средним прогнозом всех установленных FINAL сидов.
Отдельные лучшие сиды на финальном тесте не используются для научного заключения.
Временная валидация: train до development; test последний год, не участвует в выборе гиперпараметров.
Baselines: наблюдаемая инерция, среднее двух подтверждённых прошлых годов, затухающий исторический тренд; Ridge: регуляризованная линейная регрессия; CatBoost: бустинг.
Категориальная кодировка и числовая импутация обучаются только на train.
Интервалы: кластерный bootstrap по регионам или группам организаций. Holm — подтверждающие сравнения.
E0/E1: одинаковая Ridge, одинаковые муниципалитеты/годы; в E1 дополнительно число подтверждённых площадок ВО.
Миграция ради образования не равна изменению постоянного места жительства.
Рост прогностической точности не идентифицирует причинный эффект открытия колледжа или вуза.
Неподтверждённые годы публикации позволяют только ретроспективную проверку при предположении о лагах.
"""

def parquet_export(name,data):
    """Безопасная сериализация без изменения PANELS и числовых метрик."""
    prepared=data.copy()

    # Смешанные 2025 / historical должны иметь единый тип.
    if name=="group_protocols" and "test_year" in prepared:
        prepared["test_year"]=prepared["test_year"].astype("string")

    for col in prepared.select_dtypes(include=['object']).columns:
        series=prepared[col].dropna()
        types={type(v) for v in series.head(2000)}
        if len(types)>1 and (str in types or any(t in (dict,list,tuple,set) for t in types)):
            prepared[col]=prepared[col].map(lambda v:json.dumps(v,ensure_ascii=False,default=str) if isinstance(v,(dict,list,tuple,set)) else str(v) if pd.notna(v) else None)
    path=OUT/"derived"/(name+".parquet")
    path.parent.mkdir(parents=True,exist_ok=True)
    try:
        prepared.to_parquet(path,index=False,engine="pyarrow")
        check=pd.read_parquet(path)
        if len(check)!=len(prepared):
            raise RuntimeError(f"{name}: нарушено число строк")
        if list(check.columns)!=list(prepared.columns):
            raise RuntimeError(f"{name}: нарушена схема столбцов")
        return path
    except (pa.ArrowException,TypeError,ValueError) as ex:
        path.unlink(missing_ok=True)
        raise RuntimeError(f"Ошибка Parquet [{name}]: {str(ex)[:250]}") from ex

EVIDENCE.unlink(missing_ok=True)
PRED=pd.concat(PREDICTIONS,ignore_index=True) if PREDICTIONS else pd.DataFrame()

with zipfile.ZipFile(EVIDENCE,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    z.writestr("research_methods.md",METHODS)
    z.writestr("metadata/config.json",json.dumps(asdict(CFG),ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/experiment_plan.json",json.dumps(EXPERIMENT_PLAN,ensure_ascii=False,indent=2))
    z.writestr("metadata/provenance.json",json.dumps(PROVENANCE,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/qa.json",json.dumps(QA,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/modules.json",json.dumps(STATUS,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/limitations.json",json.dumps(ISSUES,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/model_parameters.json",json.dumps(MODELS_META,ensure_ascii=False,default=str,indent=2))
    z.writestr("metadata/splits.json",json.dumps(SPLITS,ensure_ascii=False,default=str,indent=2))

    for name,data in PANELS.items():
        if isinstance(data,pd.DataFrame) and not data.empty:
            path=parquet_export(name,data)
            z.write(path,"panels/"+path.name)

    if not PRED.empty:
        predpath=parquet_export("education_predictions",PRED)
        z.write(predpath,"predictions/education_predictions.parquet")

    for path in FIGURES:
        z.write(path,"figures/"+Path(path).name)
    for name,path in MODEL_FILES.items():
        z.write(path,"models/"+name)

    if (OUT/"traceback.log").exists():
        z.write(OUT/"traceback.log","run.log")
    else:
        z.writestr("run.log","No unhandled exceptions recorded in this execution.\n")

BAR.update(1)
print("Локальный экспорт PASS:",
      EXCEL.name,round(EXCEL.stat().st_size/1e6,2),"MB;",
      EVIDENCE.name,round(EVIDENCE.stat().st_size/1e6,2),"MB")


## 14–15. Контроль целостности и публикация в отдельной ветке

Сам ноутбук восстанавливается **из встроенного состава ячеек**, без Google Drive, внешнего `.py` и ручного копирования. Финальная выгрузка проходит `nbformat.validate`, сканирование секретов, проверку Excel, ZIP и каждого Parquet. `SMOKE` неизменно запрещает итоговую публикацию. При `FULL QA` данные публикуются **атомарной заменой трёх файлов в существующем сегменте собственной ветки, без force push и без изменения main**, тяжёлый evidence — в Release того же приватного репозитория; опубликованные байты скачиваются повторно и сверяются с локальными SHA-256. Если публикация не подтверждена — `PUBLICATION_FAIL`, а не фиктивный `PASS_GITHUB`.

**Замена:** целевая ветка — `ml05-education-20261102T160726Z`, прежний манифест должен иметь версию `stage2-ml5-v2.0.0`. Старый Release остаётся исторической копией; новый Release создаётся с тегом `stage2-ml5-v2.0.2-...`. Если ветка уже обновлена или контрольный SHA не совпадает, запись останавливается.


In [ ]:
# Финальная ячейка: self-export, QA и атомарная замена собственного GitHub-сегмента.
# Репозиторий Stage I и main никогда не меняются. В SMOKE запись полностью отключена.
_BUNDLE_B64 = '...'
_SELF_TEMPLATE = "# Финальная ячейка: self-export, QA и атомарная замена собственного GitHub-сегмента.\n# Репозиторий Stage I и main никогда не меняются. В SMOKE запись полностью отключена.\n_BUNDLE_B64 = __BUNDLE_LITERAL__\n_SELF_TEMPLATE = __TEMPLATE_LITERAL__\nimport io, zlib, zipfile\n\ndef _compose_self(template, bundle):\n    lines=[]\n    for line in template.splitlines():\n        if line.startswith('_BUNDLE_B64 = __BUNDLE_LITERAL__'):\n            line='_BUNDLE_B64 = '+repr(bundle)\n        elif line.startswith('_SELF_TEMPLATE = __TEMPLATE_LITERAL__'):\n            line='_SELF_TEMPLATE = '+repr(template)\n        lines.append(line)\n    return '\\n'.join(lines)+'\\n'\n\nprefix=json.loads(zlib.decompress(base64.b64decode(_BUNDLE_B64)))\nself_code=_compose_self(_SELF_TEMPLATE,_BUNDLE_B64)\nclean=nbformat.v4.new_notebook(cells=[nbformat.from_dict(c) for c in prefix]+[nbformat.v4.new_code_cell(self_code)],\n    metadata={'kernelspec':{'display_name':'Python 3','language':'python','name':'python3'},\n              'language_info':{'name':'python'},'colab':{'provenance':[]}})\nfor cell in clean.cells:\n    if cell.cell_type=='code':cell.outputs=[];cell.execution_count=None\nnbformat.validate(clean)\nnbformat.write(clean,str(NOTEBOOK))\nnt=NOTEBOOK.read_text(encoding='utf-8')\nif TOKEN in nt:raise RuntimeError('SECURITY_FAIL: секрет в ноутбуке')\nif re.search(r'(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}',nt):raise RuntimeError('SECURITY_FAIL: похоже на hardcoded token')\nassert len(clean.cells)>=28 and NOTEBOOK.stat().st_size<2*1024*1024\n\n# QA: не только формат, но и предметная инвариантность межрегиональных потоков.\nassert not od_inter.empty and len(od)==len(od_inter)+len(od_intra)+len(od_unknown)\nassert (od_inter.origin_region!=od_inter.destination_region).all()\nassert not unknown_region(od_inter.origin_region).any() and not unknown_region(od_inter.destination_region).any()\nassert all(CANARY_RESULTS.get(k,False) for k in ('ML-5.1','ML-5.3') if k in TASK_DATA)\nwith zipfile.ZipFile(EXCEL) as z:\n    assert z.testzip() is None,'Excel повреждён'\n    sheets=[n for n in z.namelist() if re.fullmatch(r'xl/worksheets/sheet\\d+\\.xml',n)]\n    assert len(sheets)>=8,'Excel слишком пустой'\n    assert all(b'<row' in z.read(n) for n in sheets),'В Excel есть пустой лист'\nwith zipfile.ZipFile(EVIDENCE) as z:\n    assert z.testzip() is None,'Повреждён ZIP'\n    required=['panels/edu_od_interregional.parquet','panels/edu_od_intraregional.parquet',\n              'panels/od_composition.parquet','panels/metrics.parquet',\n              'predictions/education_predictions.parquet','panels/canary_tests.parquet','panels/confirmatory_results.parquet','panels/od_edge_coverage.parquet','panels/holm_confirmatory.parquet']\n    assert all(n in z.namelist() for n in required),'Не хватает обязательных доказательных таблиц'\n    ZIP_STATS={}\n    for n in [n for n in z.namelist() if n.endswith('.parquet')]:\n        par=pq.ParquetFile(io.BytesIO(z.read(n)))\n        assert par.metadata.num_rows>0, 'Пустой Parquet '+n\n        ZIP_STATS[n]={'rows':int(par.metadata.num_rows),'columns':int(par.metadata.num_columns)}\nassert not PRED.empty and (PRED.target_year>PRED.origin_year).all()\nkey=[c for c in ['module','entity_id','origin_year','target_year','split_id','model','seed'] if c in PRED.columns]\nassert not PRED.duplicated(key).any(), 'Дубли прогнозов'\np1=PRED[PRED.module=='ML-5.1']\nassert len(p1)>0 and (p1.origin_region!=p1.destination_region).all(), 'Петли в прогнозах ML-5.1'\npo=PRED[PRED.module=='ML-5.3'].drop_duplicates(['entity_id','target_year'])\nref=PANELS['outflow_interregional_observed'].rename(columns={'year':'target_year',\n          'interregional_outflow_observed':'expected_total'})\nif len(po):\n    chk=po[['origin_region','sector','target_year','actual']].merge(ref,\n       on=['origin_region','sector','target_year'],how='left',validate='many_to_one')\n    assert chk.expected_total.notna().all() and np.allclose(chk.actual,chk.expected_total), 'ML-5.3 не равен настоящему межрегиональному агрегату'\nassert all((p['status'] in ['TESTED_FULL','DESCRIPTIVE_PASS','NOT_FEASIBLE'])\n           for p in STATUS.values()) if CFG.MODE=='FULL' else True\nassert any(v['status']=='TESTED_FULL' for v in STATUS.values()) if CFG.MODE=='FULL' else True\n\nINPUT_HASHES={}\nfor a in ASSETS:\n    p=OUT/'inputs'/(hashlib.sha256(a['asset_id'].encode()).hexdigest()[:20]+'.parquet')\n    if p.exists():\n        assert sha256(p)==a['sha256'], 'Потеря SHA Stage I '+a['asset_id']\n        INPUT_HASHES[a['asset_id']]=a['sha256']\nOUTPUT_HASHES={p.name:sha256(p) for p in [EXCEL,EVIDENCE,NOTEBOOK]}\nFULL_QA=(CFG.MODE=='FULL' and bool(len(METRICS)) and len(ZIP_STATS)>=6\n         and all(CANARY_RESULTS.values()) and len(p1)>0 and len(po)>0\n         and isinstance(PANELS.get('confirmatory_results'),pd.DataFrame)\n         and all(m in set(PANELS['confirmatory_results'].module) for m in ('ML-5.1','ML-5.3')))\nRUN_MANIFEST={\n  'version':CFG.VERSION,'mode':CFG.MODE,'run_status':'FULL_LOCAL_QA_PASS_WITH_LIMITATIONS' if FULL_QA else 'SMOKE_ONLY',\n  'git_publication':'NOT_ATTEMPTED','repo':CFG.REPO,'segment':CFG.SEGMENT,\n  'supersedes_version':CFG.EXPECT_OLD_VERSION,'target_branch':CFG.TARGET_BRANCH,\n  'stage1_index_version':INDEX['version'],\n  'input_sha256':INPUT_HASHES,'outputs_sha256':OUTPUT_HASHES,\n  'plan_hash':PLAN_HASH,'parquet_zip':ZIP_STATS,'module_status':STATUS,\n  'qa':QA,'canary_results':CANARY_RESULTS,'od_composition':OD_INTER_PRE_FLIGHT,\n  'confirmatory_results':PANELS.get('confirmatory_results',pd.DataFrame()).to_dict('records'),'model_metrics_count':len(METRICS),'predictions_rows':len(PRED),\n  'scientific_caveat':'Неполный universe пар; только 3/7 модулей имеют научные оценки. OD по E01 не доказывает постоянную смену места жительства; 4 модуля могут иметь NOT_FEASIBLE; расстояния и экономические эффекты H5.1 не проверены',\n  'elapsed_seconds':round(time.monotonic()-RUN_STARTED,1),\n  'generated_utc':datetime.now(timezone.utc).isoformat()\n}\nMANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')\nBAR.update(1)\n\ndef gh(method,api_path,**kwargs):\n    r=safe_request(method,API+api_path,**kwargs)\n    try:return r.json()\n    finally:r.close()\n\ndef remote_sha256_file(branch,path):\n    url='https://raw.githubusercontent.com/'+CFG.REPO+'/'+quote(branch,safe='/')+'/'+quote(path,safe='/')\n    r=safe_request('GET',url,stream=True,timeout=(30,900))\n    try:\n        if 'json' in r.headers.get('Content-Type','').lower():raise RuntimeError('GitHub raw вернул JSON вместо файла')\n        h=hashlib.sha256()\n        for chunk in r.iter_content(4*1024*1024):\n            if chunk:h.update(chunk)\n        return h.hexdigest()\n    finally:r.close()\n\ndef remote_sha256_release(asset_id):\n    r=safe_request('GET',API+'/releases/assets/'+str(asset_id),\n                   headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))\n    if 'json' in r.headers.get('Content-Type','').lower():\n        d=r.json();r.close()\n        r=safe_request('GET',d['browser_download_url'],stream=True,timeout=(30,900))\n    try:\n        h=hashlib.sha256()\n        for chunk in r.iter_content(4*1024*1024):\n            if chunk:h.update(chunk)\n        return h.hexdigest()\n    finally:r.close()\n\ndef atomically_replace_own_files(branch, file_paths, expected_head):\n    # Создаём единый Git tree с заменой трёх файлов, один commit и fast-forward без force.\n    assert CFG.SEGMENT=='stage2/ml05_education' and branch==CFG.TARGET_BRANCH and branch!='main'\n    head=gh('GET','/git/ref/heads/'+quote(branch,safe='/'))['object']['sha']\n    if head!=expected_head:raise RuntimeError('CONCURRENT_MODIFICATION: ветка обновлена другим процессом')\n    base=gh('GET','/git/commits/'+head)['tree']['sha']\n    entries=[]\n    for file in file_paths:\n        blob=gh('POST','/git/blobs',json={'content':base64.b64encode(file.read_bytes()).decode('ascii'),'encoding':'base64'},timeout=(30,900))\n        entries.append({'path':CFG.SEGMENT+'/'+file.name,'mode':'100644','type':'blob','sha':blob['sha']})\n    tree=gh('POST','/git/trees',json={'base_tree':base,'tree':entries})\n    commit=gh('POST','/git/commits',json={'message':'ML-05 v2.0.2: исправлены OD, контроль утечек, научная валидация',\n                      'tree':tree['sha'],'parents':[head]})\n    check=gh('GET','/git/ref/heads/'+quote(branch,safe='/'))['object']['sha']\n    if check!=head:raise RuntimeError('CONCURRENT_MODIFICATION: ветка изменилась перед commit')\n    gh('PATCH','/git/refs/heads/'+quote(branch,safe='/'),json={'sha':commit['sha'],'force':False})\n    return commit['sha']\n\ndef update_manifest_only(branch,expected_head):\n    path=CFG.SEGMENT+'/run_manifest.json'\n    r=gh('GET','/contents/'+quote(path,safe='/'),params={'ref':branch})\n    payload={'message':'ML-05 v2.0.2: подтверждены Release SHA-256 и публикация',\n       'content':base64.b64encode(MANIFEST.read_bytes()).decode('ascii'),\n       'branch':branch,'sha':r['sha']}\n    current=gh('GET','/git/ref/heads/'+quote(branch,safe='/'))['object']['sha']\n    if current!=expected_head:raise RuntimeError('CONCURRENT_MODIFICATION: итоговый manifest заблокирован')\n    return gh('PUT','/contents/'+quote(path,safe='/'),json=payload,timeout=(30,900))['commit']['sha']\n\ndef overwrite_previous_ml5_segment():\n    if not (FULL_QA and CFG.MODE=='FULL' and CFG.OVERWRITE_OWN_SEGMENT):\n        raise RuntimeError('PUBLICATION_BLOCKED: нужен FULL QA и разрешение замены сегмента')\n    assert CFG.REPO=='AntonDozhdikov/Demograf_sovetnik'\n    assert CFG.TARGET_BRANCH=='ml05-education-20261102T160726Z'\n    head=gh('GET','/git/ref/heads/'+quote(CFG.TARGET_BRANCH,safe='/'))['object']['sha']\n    prev=gh('GET','/contents/'+quote(CFG.SEGMENT+'/run_manifest.json',safe='/'),params={'ref':CFG.TARGET_BRANCH})\n    if not prev.get('content'):raise RuntimeError('OLD_SEGMENT_NOT_VERIFIED: нет старого манифеста')\n    prev_meta=json.loads(base64.b64decode(prev['content']))\n    if prev_meta.get('version')!=CFG.EXPECT_OLD_VERSION:\n        raise RuntimeError('VERSION_CONFLICT: старый сегмент уже обновлён или не соответствует ожидаемому')\n    if prev_meta.get('segment')!=CFG.SEGMENT:\n        raise RuntimeError('PATH_GUARD: старый сегмент отличается')\n    assert EVIDENCE.stat().st_size<=CFG.MAX_ASSET_MB*1024**2, 'Release asset превышает лимит'\n    suffix=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')\n    tag=CFG.VERSION+'-'+suffix\n    RUN_MANIFEST.update({'git_publication':'PENDING_RELEASE_VERIFICATION','run_status':'PUBLISHING',\n      'previous_segment_head_sha':head,'publication_branch':CFG.TARGET_BRANCH,'release_tag':tag})\n    MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')\n    commit=atomically_replace_own_files(CFG.TARGET_BRANCH,[NOTEBOOK,EXCEL,MANIFEST],head)\n    # Ни один файл за пределами segment не изменяется. Старый тег оставлен для аудита.\n    rel=gh('POST','/releases',json={'tag_name':tag,'target_commitish':commit,\n               'name':'ML-05 · '+tag,'draft':False,'prerelease':True,\n               'generate_release_notes':False,\n               'body':'ML-05 v2.0.2: межрегиональный OD без диагонали; replaces current files only in stage2/ml05_education. Legacy tag preserved for audit.'})\n    upload_url='https://uploads.github.com/repos/'+CFG.REPO+'/releases/'+str(rel['id'])+'/assets'\n    response=safe_request('POST',upload_url,params={'name':EVIDENCE.name},data=EVIDENCE.read_bytes(),\n              headers={'Content-Type':'application/zip'},timeout=(30,1800))\n    try:asset=response.json()\n    finally:response.close()\n    assert remote_sha256_release(asset['id'])==OUTPUT_HASHES[EVIDENCE.name], 'Release checksum mismatch'\n    for file in (NOTEBOOK,EXCEL):\n        assert remote_sha256_file(CFG.TARGET_BRANCH,CFG.SEGMENT+'/'+file.name)==OUTPUT_HASHES[file.name], ('GitHub file mismatch: '+file.name)\n    RUN_MANIFEST.update({'git_publication':'PASS_GITHUB','run_status':'FULL_PASS_WITH_DOCUMENTED_LIMITATIONS',\n        'artifact_commit_sha':commit,'release_asset_url':asset['browser_download_url'],\n        'notebook_url':'https://github.com/'+CFG.REPO+'/blob/'+CFG.TARGET_BRANCH+'/'+CFG.SEGMENT+'/'+NOTEBOOK.name,\n        'excel_url':'https://github.com/'+CFG.REPO+'/blob/'+CFG.TARGET_BRANCH+'/'+CFG.SEGMENT+'/'+EXCEL.name})\n    MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')\n    final_sha=update_manifest_only(CFG.TARGET_BRANCH,commit)\n    assert remote_sha256_file(CFG.TARGET_BRANCH,CFG.SEGMENT+'/run_manifest.json')==sha256(MANIFEST)\n    return {'branch':CFG.TARGET_BRANCH,'artifact_commit':commit,'manifest_commit':final_sha,\n            'tag':tag,'release':rel['html_url']}\n\nPUBLISHED=None\nif FULL_QA and CFG.PUBLISH_AFTER_FULL_QA and CFG.OVERWRITE_OWN_SEGMENT:\n    try:PUBLISHED=overwrite_previous_ml5_segment()\n    except Exception as error:\n        RUN_MANIFEST['git_publication']='PUBLICATION_FAIL'\n        RUN_MANIFEST['run_status']='PUBLICATION_FAIL'\n        RUN_MANIFEST['publication_error_type']=type(error).__name__\n        MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')\n        (OUT/'traceback.log').write_text(traceback.format_exc(),encoding='utf-8')\n        raise RuntimeError('PUBLICATION_FAIL: GitHub-замена или сверка не завершена; смотрите локальный traceback.log') from None\nelse:print('Публикация не начата: SMOKE / недостаточная научная приёмка / запрет записи.')\nBAR.update(1);BAR.close()\nprint('\\nML-05 ·',RUN_MANIFEST['run_status'],'· реальные межрегиональные OD',len(od_inter),'· диагональ',len(od_intra))\nprint('Модульные статусы:',{k:v['status'] for k,v in STATUS.items()})\nprint('Финальные test-строки:',len(PRED),'· файл метрик:',len(METRICS),'· графиков:',len(FIGURES))\nif PUBLISHED:print('Заменён сегмент:',PUBLISHED)\nprint('Локальные выходы:',NOTEBOOK,EXCEL,EVIDENCE,MANIFEST)\n"
import io, zlib, zipfile

def _compose_self(template, bundle):
    lines=[]
    for line in template.splitlines():
        if line.startswith('_BUNDLE_B64 = __BUNDLE_LITERAL__'):
            line='_BUNDLE_B64 = '+repr(bundle)
        elif line.startswith('_SELF_TEMPLATE = __TEMPLATE_LITERAL__'):
            line='_SELF_TEMPLATE = '+repr(template)
        lines.append(line)
    return '\n'.join(lines)+'\n'

prefix=json.loads(zlib.decompress(base64.b64decode(_BUNDLE_B64)))
self_code=_compose_self(_SELF_TEMPLATE,_BUNDLE_B64)
clean=nbformat.v4.new_notebook(cells=[nbformat.from_dict(c) for c in prefix]+[nbformat.v4.new_code_cell(self_code)],
    metadata={'kernelspec':{'display_name':'Python 3','language':'python','name':'python3'},
              'language_info':{'name':'python'},'colab':{'provenance':[]}})
for cell in clean.cells:
    if cell.cell_type=='code':cell.outputs=[];cell.execution_count=None
nbformat.validate(clean)
nbformat.write(clean,str(NOTEBOOK))
nt=NOTEBOOK.read_text(encoding='utf-8')
if TOKEN in nt:raise RuntimeError('SECURITY_FAIL: секрет в ноутбуке')
if re.search(r'(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}',nt):raise RuntimeError('SECURITY_FAIL: похоже на hardcoded token')
assert len(clean.cells)>=28 and NOTEBOOK.stat().st_size<2*1024*1024

# QA: не только формат, но и предметная инвариантность межрегиональных потоков.
assert not od_inter.empty and len(od)==len(od_inter)+len(od_intra)+len(od_unknown)
assert (od_inter.origin_region!=od_inter.destination_region).all()
assert not unknown_region(od_inter.origin_region).any() and not unknown_region(od_inter.destination_region).any()
assert all(CANARY_RESULTS.get(k,False) for k in ('ML-5.1','ML-5.3') if k in TASK_DATA)
with zipfile.ZipFile(EXCEL) as z:
    assert z.testzip() is None,'Excel повреждён'
    sheets=[n for n in z.namelist() if re.fullmatch(r'xl/worksheets/sheet\d+\.xml',n)]
    assert len(sheets)>=8,'Excel слишком пустой'
    assert all(b'<row' in z.read(n) for n in sheets),'В Excel есть пустой лист'
with zipfile.ZipFile(EVIDENCE) as z:
    assert z.testzip() is None,'Повреждён ZIP'
    required=['panels/edu_od_interregional.parquet','panels/edu_od_intraregional.parquet',
              'panels/od_composition.parquet','panels/metrics.parquet',
              'predictions/education_predictions.parquet','panels/canary_tests.parquet','panels/confirmatory_results.parquet','panels/od_edge_coverage.parquet','panels/holm_confirmatory.parquet']
    assert all(n in z.namelist() for n in required),'Не хватает обязательных доказательных таблиц'
    ZIP_STATS={}
    for n in [n for n in z.namelist() if n.endswith('.parquet')]:
        par=pq.ParquetFile(io.BytesIO(z.read(n)))
        assert par.metadata.num_rows>0, 'Пустой Parquet '+n
        ZIP_STATS[n]={'rows':int(par.metadata.num_rows),'columns':int(par.metadata.num_columns)}
assert not PRED.empty and (PRED.target_year>PRED.origin_year).all()
key=[c for c in ['module','entity_id','origin_year','target_year','split_id','model','seed'] if c in PRED.columns]
assert not PRED.duplicated(key).any(), 'Дубли прогнозов'
p1=PRED[PRED.module=='ML-5.1']
assert len(p1)>0 and (p1.origin_region!=p1.destination_region).all(), 'Петли в прогнозах ML-5.1'
po=PRED[PRED.module=='ML-5.3'].drop_duplicates(['entity_id','target_year'])
ref=PANELS['outflow_interregional_observed'].rename(columns={'year':'target_year',
          'interregional_outflow_observed':'expected_total'})
if len(po):
    chk=po[['origin_region','sector','target_year','actual']].merge(ref,
       on=['origin_region','sector','target_year'],how='left',validate='many_to_one')
    assert chk.expected_total.notna().all() and np.allclose(chk.actual,chk.expected_total), 'ML-5.3 не равен настоящему межрегиональному агрегату'
assert all((p['status'] in ['TESTED_FULL','DESCRIPTIVE_PASS','NOT_FEASIBLE'])
           for p in STATUS.values()) if CFG.MODE=='FULL' else True
assert any(v['status']=='TESTED_FULL' for v in STATUS.values()) if CFG.MODE=='FULL' else True

INPUT_HASHES={}
for a in ASSETS:
    p=OUT/'inputs'/(hashlib.sha256(a['asset_id'].encode()).hexdigest()[:20]+'.parquet')
    if p.exists():
        assert sha256(p)==a['sha256'], 'Потеря SHA Stage I '+a['asset_id']
        INPUT_HASHES[a['asset_id']]=a['sha256']
OUTPUT_HASHES={p.name:sha256(p) for p in [EXCEL,EVIDENCE,NOTEBOOK]}
FULL_QA=(CFG.MODE=='FULL' and bool(len(METRICS)) and len(ZIP_STATS)>=6
         and all(CANARY_RESULTS.values()) and len(p1)>0 and len(po)>0
         and isinstance(PANELS.get('confirmatory_results'),pd.DataFrame)
         and all(m in set(PANELS['confirmatory_results'].module) for m in ('ML-5.1','ML-5.3')))
RUN_MANIFEST={
  'version':CFG.VERSION,'mode':CFG.MODE,'run_status':'FULL_LOCAL_QA_PASS_WITH_LIMITATIONS' if FULL_QA else 'SMOKE_ONLY',
  'git_publication':'NOT_ATTEMPTED','repo':CFG.REPO,'segment':CFG.SEGMENT,
  'supersedes_version':CFG.EXPECT_OLD_VERSION,'target_branch':CFG.TARGET_BRANCH,
  'stage1_index_version':INDEX['version'],
  'input_sha256':INPUT_HASHES,'outputs_sha256':OUTPUT_HASHES,
  'plan_hash':PLAN_HASH,'parquet_zip':ZIP_STATS,'module_status':STATUS,
  'qa':QA,'canary_results':CANARY_RESULTS,'od_composition':OD_INTER_PRE_FLIGHT,
  'confirmatory_results':PANELS.get('confirmatory_results',pd.DataFrame()).to_dict('records'),'model_metrics_count':len(METRICS),'predictions_rows':len(PRED),
  'scientific_caveat':'Неполный universe пар; только 3/7 модулей имеют научные оценки. OD по E01 не доказывает постоянную смену места жительства; 4 модуля могут иметь NOT_FEASIBLE; расстояния и экономические эффекты H5.1 не проверены',
  'elapsed_seconds':round(time.monotonic()-RUN_STARTED,1),
  'generated_utc':datetime.now(timezone.utc).isoformat()
}
MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
BAR.update(1)

def gh(method,api_path,**kwargs):
    r=safe_request(method,API+api_path,**kwargs)
    try:return r.json()
    finally:r.close()

def remote_sha256_file(branch,path):
    url='https://raw.githubusercontent.com/'+CFG.REPO+'/'+quote(branch,safe='/')+'/'+quote(path,safe='/')
    r=safe_request('GET',url,stream=True,timeout=(30,900))
    try:
        if 'json' in r.headers.get('Content-Type','').lower():raise RuntimeError('GitHub raw вернул JSON вместо файла')
        h=hashlib.sha256()
        for chunk in r.iter_content(4*1024*1024):
            if chunk:h.update(chunk)
        return h.hexdigest()
    finally:r.close()

def remote_sha256_release(asset_id):
    r=safe_request('GET',API+'/releases/assets/'+str(asset_id),
                   headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
    if 'json' in r.headers.get('Content-Type','').lower():
        d=r.json();r.close()
        r=safe_request('GET',d['browser_download_url'],stream=True,timeout=(30,900))
    try:
        h=hashlib.sha256()
        for chunk in r.iter_content(4*1024*1024):
            if chunk:h.update(chunk)
        return h.hexdigest()
    finally:r.close()

def atomically_replace_own_files(branch, file_paths, expected_head):
    # Создаём единый Git tree с заменой трёх файлов, один commit и fast-forward без force.
    assert CFG.SEGMENT=='stage2/ml05_education' and branch==CFG.TARGET_BRANCH and branch!='main'
    head=gh('GET','/git/ref/heads/'+quote(branch,safe='/'))['object']['sha']
    if head!=expected_head:raise RuntimeError('CONCURRENT_MODIFICATION: ветка обновлена другим процессом')
    base=gh('GET','/git/commits/'+head)['tree']['sha']
    entries=[]
    for file in file_paths:
        blob=gh('POST','/git/blobs',json={'content':base64.b64encode(file.read_bytes()).decode('ascii'),'encoding':'base64'},timeout=(30,900))
        entries.append({'path':CFG.SEGMENT+'/'+file.name,'mode':'100644','type':'blob','sha':blob['sha']})
    tree=gh('POST','/git/trees',json={'base_tree':base,'tree':entries})
    commit=gh('POST','/git/commits',json={'message':'ML-05 v2.0.2: исправлены OD, контроль утечек, научная валидация',
                      'tree':tree['sha'],'parents':[head]})
    check=gh('GET','/git/ref/heads/'+quote(branch,safe='/'))['object']['sha']
    if check!=head:raise RuntimeError('CONCURRENT_MODIFICATION: ветка изменилась перед commit')
    gh('PATCH','/git/refs/heads/'+quote(branch,safe='/'),json={'sha':commit['sha'],'force':False})
    return commit['sha']

def update_manifest_only(branch,expected_head):
    path=CFG.SEGMENT+'/run_manifest.json'
    r=gh('GET','/contents/'+quote(path,safe='/'),params={'ref':branch})
    payload={'message':'ML-05 v2.0.2: подтверждены Release SHA-256 и публикация',
       'content':base64.b64encode(MANIFEST.read_bytes()).decode('ascii'),
       'branch':branch,'sha':r['sha']}
    current=gh('GET','/git/ref/heads/'+quote(branch,safe='/'))['object']['sha']
    if current!=expected_head:raise RuntimeError('CONCURRENT_MODIFICATION: итоговый manifest заблокирован')
    return gh('PUT','/contents/'+quote(path,safe='/'),json=payload,timeout=(30,900))['commit']['sha']

def overwrite_previous_ml5_segment():
    if not (FULL_QA and CFG.MODE=='FULL' and CFG.OVERWRITE_OWN_SEGMENT):
        raise RuntimeError('PUBLICATION_BLOCKED: нужен FULL QA и разрешение замены сегмента')
    assert CFG.REPO=='AntonDozhdikov/Demograf_sovetnik'
    assert CFG.TARGET_BRANCH=='ml05-education-20261102T160726Z'
    head=gh('GET','/git/ref/heads/'+quote(CFG.TARGET_BRANCH,safe='/'))['object']['sha']
    prev=gh('GET','/contents/'+quote(CFG.SEGMENT+'/run_manifest.json',safe='/'),params={'ref':CFG.TARGET_BRANCH})
    if not prev.get('content'):raise RuntimeError('OLD_SEGMENT_NOT_VERIFIED: нет старого манифеста')
    prev_meta=json.loads(base64.b64decode(prev['content']))
    if prev_meta.get('version')!=CFG.EXPECT_OLD_VERSION:
        raise RuntimeError('VERSION_CONFLICT: старый сегмент уже обновлён или не соответствует ожидаемому')
    if prev_meta.get('segment')!=CFG.SEGMENT:
        raise RuntimeError('PATH_GUARD: старый сегмент отличается')
    assert EVIDENCE.stat().st_size<=CFG.MAX_ASSET_MB*1024**2, 'Release asset превышает лимит'
    suffix=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    tag=CFG.VERSION+'-'+suffix
    RUN_MANIFEST.update({'git_publication':'PENDING_RELEASE_VERIFICATION','run_status':'PUBLISHING',
      'previous_segment_head_sha':head,'publication_branch':CFG.TARGET_BRANCH,'release_tag':tag})
    MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    commit=atomically_replace_own_files(CFG.TARGET_BRANCH,[NOTEBOOK,EXCEL,MANIFEST],head)
    # Ни один файл за пределами segment не изменяется. Старый тег оставлен для аудита.
    rel=gh('POST','/releases',json={'tag_name':tag,'target_commitish':commit,
               'name':'ML-05 · '+tag,'draft':False,'prerelease':True,
               'generate_release_notes':False,
               'body':'ML-05 v2.0.2: межрегиональный OD без диагонали; replaces current files only in stage2/ml05_education. Legacy tag preserved for audit.'})
    upload_url='https://uploads.github.com/repos/'+CFG.REPO+'/releases/'+str(rel['id'])+'/assets'
    response=safe_request('POST',upload_url,params={'name':EVIDENCE.name},data=EVIDENCE.read_bytes(),
              headers={'Content-Type':'application/zip'},timeout=(30,1800))
    try:asset=response.json()
    finally:response.close()
    assert remote_sha256_release(asset['id'])==OUTPUT_HASHES[EVIDENCE.name], 'Release checksum mismatch'
    for file in (NOTEBOOK,EXCEL):
        assert remote_sha256_file(CFG.TARGET_BRANCH,CFG.SEGMENT+'/'+file.name)==OUTPUT_HASHES[file.name], ('GitHub file mismatch: '+file.name)
    RUN_MANIFEST.update({'git_publication':'PASS_GITHUB','run_status':'FULL_PASS_WITH_DOCUMENTED_LIMITATIONS',
        'artifact_commit_sha':commit,'release_asset_url':asset['browser_download_url'],
        'notebook_url':'https://github.com/'+CFG.REPO+'/blob/'+CFG.TARGET_BRANCH+'/'+CFG.SEGMENT+'/'+NOTEBOOK.name,
        'excel_url':'https://github.com/'+CFG.REPO+'/blob/'+CFG.TARGET_BRANCH+'/'+CFG.SEGMENT+'/'+EXCEL.name})
    MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    final_sha=update_manifest_only(CFG.TARGET_BRANCH,commit)
    assert remote_sha256_file(CFG.TARGET_BRANCH,CFG.SEGMENT+'/run_manifest.json')==sha256(MANIFEST)
    return {'branch':CFG.TARGET_BRANCH,'artifact_commit':commit,'manifest_commit':final_sha,
            'tag':tag,'release':rel['html_url']}

PUBLISHED=None
if FULL_QA and CFG.PUBLISH_AFTER_FULL_QA and CFG.OVERWRITE_OWN_SEGMENT:
    try:PUBLISHED=overwrite_previous_ml5_segment()
    except Exception as error:
        RUN_MANIFEST['git_publication']='PUBLICATION_FAIL'
        RUN_MANIFEST['run_status']='PUBLICATION_FAIL'
        RUN_MANIFEST['publication_error_type']=type(error).__name__
        MANIFEST.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
        (OUT/'traceback.log').write_text(traceback.format_exc(),encoding='utf-8')
        raise RuntimeError('PUBLICATION_FAIL: GitHub-замена или сверка не завершена; смотрите локальный traceback.log') from None
else:print('Публикация не начата: SMOKE / недостаточная научная приёмка / запрет записи.')
BAR.update(1);BAR.close()
print('\nML-05 ·',RUN_MANIFEST['run_status'],'· реальные межрегиональные OD',len(od_inter),'· диагональ',len(od_intra))
print('Модульные статусы:',{k:v['status'] for k,v in STATUS.items()})
print('Финальные test-строки:',len(PRED),'· файл метрик:',len(METRICS),'· графиков:',len(FIGURES))
if PUBLISHED:print('Заменён сегмент:',PUBLISHED)
print('Локальные выходы:',NOTEBOOK,EXCEL,EVIDENCE,MANIFEST)
